# Spatio-Temporal Multimodal Video RAG

This notebook implements an end-to-end video retrieval system that combines visual, speech, OCR, spatial, and temporal evidence.

The pipeline contains video ingestion, scene and speech-aware segmentation, ASR, spatial OCR, multimodal embeddings, persistent Qdrant indexing, multimodal retrieval, reranking, spatial verification, temporal reasoning, a strict FastAPI search endpoint, and performance measurement.

The strict search response contains only start_timestamp, end_timestamp, and confidence_score.

The experimental section uses real queries against the indexed video. Manual ground-truth intervals must be entered independently before reporting Temporal IoU.


In [1]:
import os
import sys
import subprocess

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = (
    "expandable_segments:True,max_split_size_mb:128"
)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["VIDEO_RAG_OCR_DEVICE"] = "cuda:0"
os.environ["VIDEO_RAG_OCR_LANGS"] = "en"
os.environ["VIDEO_RAG_OCR_FRAMES_PER_CLIP"] = "6"
os.environ["VIDEO_RAG_OCR_CANVAS_SIZE"] = "1280"
os.environ["VIDEO_RAG_QDRANT_PATH"] = "storage/qdrant_db"

PINNED = [
    "transformers==4.57.6",
    "sentence-transformers==5.6.0",
    "qdrant-client==1.18.0",
    "accelerate>=1.10,<2",
    "bitsandbytes>=0.47,<0.50",
    "faster-whisper>=1.1,<2",
    "fastapi>=0.115,<0.130",
    "uvicorn>=0.30,<1",
    "scenedetect==0.7.1",
    "easyocr>=1.7,<2",
    "SpeechRecognition>=3.10,<4",
    "psutil>=5.9,<8",
]

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "--upgrade-strategy",
    "only-if-needed",
    *PINNED,
])

import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Dependency installation completed.")
print("Restart the runtime once after changing package versions.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 89.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.4/596.4 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.2/146.2 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.2/103.2 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 58.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.5/39.5 MB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.


PyTorch version: 2.10.0+cu128
CUDA available: True
Dependency installation completed.
Restart the runtime once after changing package versions.


In [2]:
import os

directories = [
    "ingestion",
    "search",
    "api",
    "storage",
    "storage/processed",
    "monitoring",
    "evaluation",
    "runtime",
]

for directory in directories:
    os.makedirs(directory, exist_ok=True)

for package in [
    "ingestion",
    "search",
    "api",
    "storage",
    "monitoring",
    "evaluation",
    "runtime",
]:
    init_path = os.path.join(package, "__init__.py")
    if not os.path.exists(init_path):
        open(init_path, "w", encoding="utf-8").close()

print("Project directories created.")


Project directories created.


In [3]:
%%writefile ingestion/scene_detector.py
"""Content-aware scene detector with a deterministic fixed-window fallback."""
from dataclasses import dataclass
from typing import List
import os
import cv2

@dataclass
class SceneBoundary:
    start_ts: float
    end_ts: float

class SceneDetector:
    def __init__(self, target_scene_sec: float = 6.0, threshold: float = 27.0):
        self.target_scene_sec = float(target_scene_sec)
        self.threshold = float(threshold)

    def _duration(self, video_path: str) -> float:
        cap = cv2.VideoCapture(video_path)
        try:
            if not cap.isOpened():
                raise ValueError(f"Cannot open video: {video_path}")
            fps = cap.get(cv2.CAP_PROP_FPS)
            total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            if not fps or fps <= 0 or total_frames <= 0:
                raise ValueError(f"Invalid video metadata: fps={fps}, frames={total_frames}")
            return total_frames / fps
        finally:
            cap.release()

    def _fixed_windows(self, duration: float) -> List[SceneBoundary]:
        result = []
        start = 0.0
        while start < duration:
            end = min(start + self.target_scene_sec, duration)
            if end > start:
                result.append(SceneBoundary(round(start, 3), round(end, 3)))
            start = end
        return result

    def detect_scenes(self, video_path: str) -> List[SceneBoundary]:
        if not os.path.exists(video_path):
            raise FileNotFoundError(video_path)
        duration = self._duration(video_path)
        try:
            from scenedetect import detect, ContentDetector
            raw_scenes = detect(video_path, ContentDetector(threshold=self.threshold))
            scenes = [
                SceneBoundary(round(start.get_seconds(), 3), round(end.get_seconds(), 3))
                for start, end in raw_scenes
                if end.get_seconds() > start.get_seconds()
            ]
            if scenes:
                print(f"[SceneDetector] Content-aware scenes: {len(scenes)}")
                return scenes
        except Exception as exc:
            print(f"[SceneDetector] Content detector unavailable; fixed-window fallback: {exc}")
        scenes = self._fixed_windows(duration)
        print(f"[SceneDetector] Fixed-window fallback: {len(scenes)}")
        return scenes


Writing ingestion/scene_detector.py


In [4]:
%%writefile ingestion/video_processor.py
"""Adaptive scene/speech chunking plus timestamped keyframe extraction."""
import os
import shutil
import subprocess
from typing import List, Dict, Any, Optional

import cv2
import numpy as np

from .scene_detector import SceneDetector


class VideoProcessor:
    def __init__(
        self,
        min_clip: float = 1.5,
        max_clip: float = 10.0,
        dedup_gap: float = 1.25,
        frame_sample_fps: float = 1.0,
        max_keyframes_per_clip: int = 8,
        scene_detector: Optional[SceneDetector] = None,
    ):
        self.min_clip = float(min_clip)
        self.max_clip = float(max_clip)
        self.dedup_gap = float(dedup_gap)
        self.frame_sample_fps = max(float(frame_sample_fps), 0.25)
        self.max_keyframes_per_clip = max(int(max_keyframes_per_clip), 3)
        self.scene_detector = scene_detector or SceneDetector()

    @staticmethod
    def video_metadata(video_path: str) -> Dict[str, float]:
        cap = cv2.VideoCapture(video_path)
        try:
            if not cap.isOpened():
                raise ValueError(f"Cannot open video: {video_path}")
            fps = float(cap.get(cv2.CAP_PROP_FPS) or 30.0)
            frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
            width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
            height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)
            duration = frames / fps if fps > 0 and frames > 0 else 0.0
            return {
                "fps": fps,
                "frames": frames,
                "width": width,
                "height": height,
                "duration": duration,
            }
        finally:
            cap.release()

    def _build_clips(self, duration: float, scene_bounds, asr_boundaries: List[float]):
        boundaries = {0.0, round(duration, 3)}
        for scene in scene_bounds:
            boundaries.add(round(float(scene.start_ts), 3))
            boundaries.add(round(float(scene.end_ts), 3))
        for boundary in asr_boundaries:
            if 0.0 < float(boundary) < duration:
                boundaries.add(round(float(boundary), 3))

        ordered = sorted(boundaries)
        deduped = [ordered[0]]
        for boundary in ordered[1:]:
            if boundary - deduped[-1] >= self.dedup_gap:
                deduped.append(boundary)
            else:
                deduped[-1] = max(deduped[-1], boundary)
        if deduped[-1] < duration:
            deduped.append(duration)

        clips = []
        start = deduped[0]
        for end in deduped[1:]:
            while end - start > self.max_clip:
                split = start + self.max_clip
                clips.append((start, split))
                start = split
            if end - start >= self.min_clip:
                clips.append((start, end))
                start = end

        if start < duration:
            if clips and duration - start < self.min_clip:
                previous_start, _ = clips.pop()
                clips.append((previous_start, duration))
            else:
                clips.append((start, duration))
        return [(round(a, 3), round(b, 3)) for a, b in clips if b > a]

    def _sample_timestamps(self, start_ts: float, end_ts: float) -> List[float]:
        duration = max(end_ts - start_ts, 0.001)
        step = 1.0 / self.frame_sample_fps
        values = list(np.arange(start_ts, end_ts, step))
        values.extend([start_ts, start_ts + duration / 2.0, max(start_ts, end_ts - 0.05)])
        values = sorted({round(min(max(v, start_ts), end_ts - 1e-3), 3) for v in values})
        if len(values) > self.max_keyframes_per_clip:
            idx = np.linspace(0, len(values) - 1, self.max_keyframes_per_clip).round().astype(int)
            values = [values[i] for i in idx]
        return values

    def process_video(
        self, video_path: str, asr_boundaries: List[float], output_dir: str
    ) -> List[Dict[str, Any]]:
        metadata = self.video_metadata(video_path)
        duration, fps = metadata["duration"], metadata["fps"]
        scenes = self.scene_detector.detect_scenes(video_path)
        clip_bounds = self._build_clips(duration, scenes, asr_boundaries)

        os.makedirs(output_dir, exist_ok=True)
        cap = cv2.VideoCapture(video_path)
        clips_data: List[Dict[str, Any]] = []
        try:
            for idx, (start_ts, end_ts) in enumerate(clip_bounds):
                if idx % 25 == 0:
                    print(
                        f"[VideoProcessor] Extracting clip {idx + 1}/"
                        f"{len(clip_bounds)}"
                    )
                keyframes = []
                for frame_order, timestamp in enumerate(self._sample_timestamps(start_ts, end_ts)):
                    frame_index = int(round(timestamp * fps))
                    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_index)
                    ok, frame = cap.read()
                    if not ok:
                        continue
                    height, width = frame.shape[:2]
                    path = os.path.join(output_dir, f"kf_{idx}_{frame_order}_{timestamp:.3f}.jpg")
                    if not cv2.imwrite(
                        path,
                        frame,
                        [int(cv2.IMWRITE_JPEG_QUALITY), 88],
                    ):
                        del frame
                        continue
                    keyframes.append({
                        "frame_id": f"clip_{idx}_frame_{frame_order}",
                        "path": path,
                        "timestamp": float(timestamp),
                        "frame_index": frame_index,
                        "width": int(width),
                        "height": int(height),
                        "order": frame_order,
                    })
                    del frame

                if idx % 25 == 0:
                    import gc
                    gc.collect()

                clips_data.append({
                    "clip_id": f"clip_{idx}",
                    "scene_id": f"scene_{idx}",
                    "start_ts": float(start_ts),
                    "end_ts": float(end_ts),
                    "duration": round(end_ts - start_ts, 3),
                    "source_video_path": video_path,
                    "keyframes": keyframes,
                    "keyframe_paths": [item["path"] for item in keyframes],
                    "keyframe_timestamps": [item["timestamp"] for item in keyframes],
                })
        finally:
            cap.release()
        return clips_data

    def extract_audio(self, video_path_or_url: str, output_audio_path: str) -> str:
        os.makedirs(os.path.dirname(output_audio_path), exist_ok=True)
        ffmpeg_bin = shutil.which("ffmpeg") or "ffmpeg"
        command = [
            ffmpeg_bin, "-y", "-i", video_path_or_url,
            "-vn", "-acodec", "pcm_s16le", "-ar", "16000", "-ac", "1",
            output_audio_path,
        ]
        subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True)
        print(f"[Audio Extraction]  {output_audio_path}")
        return output_audio_path


Writing ingestion/video_processor.py


In [5]:
%%writefile ingestion/transcriber.py
"""Lazy, explicitly placed Whisper transcriber with deterministic release."""
import concurrent.futures
import gc
import math
import os
import re
from typing import List, Dict, Any, Optional

from search.device_detector import configured_devices, release_cuda


class AudioTranscriber:
    def __init__(
        self,
        model_size: str = "large-v3-turbo",
        window_sec: float = 10.0,
        languages: Optional[List[str]] = None,
        device: Optional[str] = None,
        autoload: bool = False,
    ):
        self.model_size = model_size
        self.window_sec = float(window_sec)
        self.languages = languages or ["ar-SA", "en-US"]
        self.device = device or configured_devices()["ingest"]
        self.model = None
        self.backend = "none"
        if autoload:
            self.load_model()

    def load_model(self):
        if self.model is not None:
            return self
        if self.device.startswith("cuda"):
            device_type = "cuda"
            device_index = int(self.device.split(":")[-1])
            compute_type = "float16"
        else:
            device_type = "cpu"
            device_index = 0
            compute_type = "int8"

        try:
            from faster_whisper import WhisperModel
            print(
                f"[AudioTranscriber] Loading Whisper {self.model_size} "
                f"on {self.device} ({compute_type})..."
            )
            self.model = WhisperModel(
                self.model_size,
                device=device_type,
                device_index=device_index,
                compute_type=compute_type,
            )
            self.backend = "faster_whisper"
            print(f"[AudioTranscriber]  Loaded on {self.device}")
            return self
        except Exception as exc:
            print(f"[AudioTranscriber] Large model unavailable: {exc}")

        try:
            from faster_whisper import WhisperModel
            self.model = WhisperModel("base", device="cpu", compute_type="int8")
            self.backend = "faster_whisper_cpu"
            self.device = "cpu"
            print("[AudioTranscriber]  CPU base fallback loaded")
        except Exception as exc:
            print(f"[AudioTranscriber] No local Whisper model: {exc}")
            self.model = None
        return self

    def release_model(self):
        if self.model is not None:
            try:
                del self.model
            except Exception:
                pass
        self.model = None
        self.backend = "released"
        gc.collect()
        release_cuda(self.device)
        print(f"[AudioTranscriber]  Released model from {self.device}")

    def _recognize_chunk(self, audio_path, start_sec, chunk_len):
        import speech_recognition as sr
        recognizer = sr.Recognizer()
        output = []
        try:
            with sr.AudioFile(audio_path) as source:
                audio_data = recognizer.record(source, offset=start_sec, duration=chunk_len)
            text = ""
            for language in self.languages:
                try:
                    text = recognizer.recognize_google(audio_data, language=language)
                    if text:
                        break
                except Exception:
                    continue
            words = text.split()
            duration = chunk_len / max(len(words), 1)
            for index, word in enumerate(words):
                start = start_sec + index * duration
                output.append({
                    "word": word.strip(),
                    "start": round(start, 3),
                    "end": round(start + duration, 3),
                })
        except Exception:
            pass
        return output

    def transcribe_audio(self, audio_path):
        if not os.path.exists(audio_path):
            return []
        self.load_model()
        words = []

        if self.model is not None:
            try:
                segments, _ = self.model.transcribe(
                    audio_path,
                    word_timestamps=True,
                    vad_filter=True,
                )
                for segment in segments:
                    for word in segment.words or []:
                        words.append({
                            "word": word.word.strip(),
                            "start": round(float(word.start), 3),
                            "end": round(float(word.end), 3),
                        })
                if words:
                    return words
            except Exception as exc:
                print(f"[AudioTranscriber] Local transcription fallback: {exc}")

        try:
            import wave
            with wave.open(audio_path, "rb") as handle:
                duration = handle.getnframes() / float(handle.getframerate())
            chunks = []
            for index in range(math.ceil(duration / self.window_sec)):
                start = index * self.window_sec
                length = min(self.window_sec, duration - start)
                if length > 0.5:
                    chunks.append((start, length))
            with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
                futures = [
                    executor.submit(self._recognize_chunk, audio_path, start, length)
                    for start, length in chunks
                ]
                for future in concurrent.futures.as_completed(futures):
                    words.extend(future.result())
            return sorted(words, key=lambda item: item["start"])
        except Exception as exc:
            print(f"[AudioTranscriber] ASR fallback failed: {exc}")
            return []

    @staticmethod
    def detect_sentence_boundaries(word_timestamps):
        boundaries = []
        punctuation = re.compile(r"[.?!;؛؟]")
        for index, word in enumerate(word_timestamps):
            if punctuation.search(word["word"]):
                boundaries.append(word["end"])
            elif index + 1 < len(word_timestamps):
                if word_timestamps[index + 1]["start"] - word["end"] > 1.5:
                    boundaries.append(word["end"])
        return sorted(set(boundaries))

    @staticmethod
    def align_transcripts_to_chunks(clips, word_timestamps):
        for clip in clips:
            start = float(clip.get("start_ts", 0.0))
            end = float(clip.get("end_ts", 0.0))
            selected = [
                word for word in word_timestamps
                if start <= float(word["start"]) < end
                or start < float(word["end"]) <= end
            ]
            clip["word_timestamps"] = selected
            clip["transcript"] = " ".join(item["word"] for item in selected).strip()
        return clips


Writing ingestion/transcriber.py


In [6]:
%%writefile ingestion/ocr_extractor.py
"""Memory-stable spatial OCR for Colab T4 x2.

EasyOCR is the default because it can be pinned explicitly to cuda:1 and
loaded independently from the ingestion models on cuda:0. PaddleOCR is not
loaded in this notebook, preventing PP-OCRv6 pipeline initialization from
killing the Colab kernel.
"""
import gc
import os
import re
from difflib import SequenceMatcher
from typing import List, Dict, Optional, Any, Tuple

try:
    import torch
except Exception:
    torch = None

from PIL import Image


class OCRExtractor:
    def __init__(
        self,
        languages: List[str] = None,
        device: Optional[str] = None,
        autoload: bool = False,
        max_frames_per_clip: int = 6,
        canvas_size: int = 1280,
    ):
        self.languages = languages or ["en"]
        self.device = device or os.environ.get("VIDEO_RAG_OCR_DEVICE", "cpu")
        self.max_frames_per_clip = max(1, int(max_frames_per_clip))
        self.canvas_size = max(640, int(canvas_size))
        self.backend = "none"
        self.reader = None
        self.processed_images = 0
        if autoload:
            self.load_model()

    def load_model(self):
        if self.reader is not None:
            return self

        requested_device = str(self.device)
        print(
            f"[OCRExtractor] Loading EasyOCR; requested={requested_device}, "
            f"languages={self.languages}..."
        )
        try:
            import easyocr

            gpu_argument: Any = False
            if requested_device.startswith("cuda"):
                if torch is None or not torch.cuda.is_available():
                    print("[OCRExtractor] CUDA unavailable; using CPU")
                    gpu_argument = False
                    self.device = "cpu"
                else:
                    # EasyOCR wraps its detector/recognizer with torch.nn.DataParallel.
                    # DataParallel requires the source module on cuda:0. Passing cuda:1
                    # produces: "module must have its parameters ... on cuda:0".
                    # OCR runs after Whisper is released, so cuda:0 is free here.
                    requested_index = int(requested_device.split(":")[-1])
                    if requested_index != 0:
                        print(
                            f"[OCRExtractor] EasyOCR DataParallel cannot use "
                            f"{requested_device} as its source device; remapping to cuda:0"
                        )
                    self.device = "cuda:0"
                    torch.cuda.set_device(0)
                    # True lets EasyOCR choose its supported primary CUDA path.
                    gpu_argument = True

            self.reader = easyocr.Reader(
                self.languages,
                gpu=gpu_argument,
                verbose=False,
                quantize=(gpu_argument is False),
                cudnn_benchmark=False,
            )
            self.backend = "easyocr"
            print(
                f"[OCRExtractor]  EasyOCR loaded; "
                f"effective_device={self.device}, gpu_argument={gpu_argument}"
            )
        except Exception as exc:
            self.reader = None
            self.backend = "none"
            raise RuntimeError(f"EasyOCR could not be loaded: {exc}") from exc
        return self

    def release_model(self):
        if self.reader is not None:
            try:
                del self.reader
            except Exception:
                pass
        self.reader = None
        self.backend = "released"
        gc.collect()

        if torch is not None and torch.cuda.is_available():
            try:
                if str(self.device).startswith("cuda"):
                    index = int(str(self.device).split(":")[-1])
                    with torch.cuda.device(index):
                        torch.cuda.synchronize()
                        torch.cuda.empty_cache()
                        torch.cuda.ipc_collect()
            except Exception:
                torch.cuda.empty_cache()
        print(f"[OCRExtractor]  Released OCR from {self.device}")

    @staticmethod
    def _clean(text: str) -> str:
        return " ".join(str(text).split()).strip()

    @staticmethod
    def _box_from_polygon(points: Any) -> Optional[List[int]]:
        try:
            xs = [float(point[0]) for point in points]
            ys = [float(point[1]) for point in points]
            return [
                int(round(min(xs))),
                int(round(min(ys))),
                int(round(max(xs))),
                int(round(max(ys))),
            ]
        except Exception:
            return None

    @staticmethod
    def _bbox_norm(bbox: List[int], width: int, height: int) -> List[int]:
        if not bbox or width <= 0 or height <= 0:
            return []
        x1, y1, x2, y2 = bbox
        return [
            int(round(1000 * x1 / width)),
            int(round(1000 * y1 / height)),
            int(round(1000 * x2 / width)),
            int(round(1000 * y2 / height)),
        ]

    @staticmethod
    def _dedupe_regions(regions: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        best: Dict[Tuple[str, Tuple[int, ...]], Dict[str, Any]] = {}
        for region in regions:
            key = (
                region.get("text", "").casefold(),
                tuple(region.get("bbox_norm", []) or []),
            )
            if key not in best or region.get("confidence", 0.0) > best[key].get("confidence", 0.0):
                best[key] = region
        return list(best.values())

    def extract_regions_from_frame(
        self,
        image_path: str,
        timestamp: float = 0.0,
        frame_id: str = "",
    ) -> Dict[str, Any]:
        if not os.path.exists(image_path):
            return {
                "frame_id": frame_id,
                "path": image_path,
                "timestamp": float(timestamp),
                "text": "",
                "regions": [],
                "width": 0,
                "height": 0,
            }

        self.load_model()

        with Image.open(image_path) as image:
            width, height = image.size

        regions: List[Dict[str, Any]] = []
        try:
            result = self.reader.readtext(
                image_path,
                decoder="greedy",
                beamWidth=1,
                batch_size=1,
                workers=0,
                detail=1,
                paragraph=False,
                min_size=8,
                text_threshold=0.55,
                low_text=0.30,
                link_threshold=0.30,
                canvas_size=self.canvas_size,
                mag_ratio=1.0,
                rotation_info=None,
            )
            for item in result:
                if len(item) < 3:
                    continue
                polygon, text, confidence = item[0], item[1], float(item[2])
                clean = self._clean(text)
                if not clean or confidence < 0.18:
                    continue
                bbox = self._box_from_polygon(polygon) or []
                regions.append({
                    "text": clean,
                    "confidence": confidence,
                    "bbox": bbox,
                    "bbox_norm": self._bbox_norm(bbox, width, height) if bbox else [],
                })
        except RuntimeError as exc:
            error_text = str(exc).casefold()
            if "module must have its parameters" in error_text:
                raise RuntimeError(
                    "EasyOCR DataParallel device mismatch. The OCR engine must "
                    "start on cuda:0; use Video RAG 3.4 or set "
                    "VIDEO_RAG_OCR_DEVICE=cuda:0 before importing the project."
                ) from exc

            # Retry once on CPU if a CUDA allocation fails.
            if "out of memory" in error_text and str(self.device).startswith("cuda"):
                print("[OCRExtractor] CUDA OOM; retrying this frame on CPU")
                self.release_model()
                self.device = "cpu"
                self.load_model()
                return self.extract_regions_from_frame(
                    image_path,
                    timestamp=timestamp,
                    frame_id=frame_id,
                )
            raise

        self.processed_images += 1
        if self.processed_images % 50 == 0:
            gc.collect()
            if torch is not None and torch.cuda.is_available():
                try:
                    index = int(str(self.device).split(":")[-1])
                    with torch.cuda.device(index):
                        torch.cuda.empty_cache()
                except Exception:
                    pass
            print(f"[OCRExtractor] Processed {self.processed_images} frames")

        regions = self._dedupe_regions(regions)
        text = " ".join(dict.fromkeys(
            region["text"] for region in regions if region.get("text")
        ))
        return {
            "frame_id": frame_id,
            "path": image_path,
            "timestamp": float(timestamp),
            "text": text,
            "regions": regions,
            "width": int(width),
            "height": int(height),
        }

    @staticmethod
    def _similarity(a: str, b: str) -> float:
        a_tokens = set(re.findall(r"[\w-]+", a.casefold()))
        b_tokens = set(re.findall(r"[\w-]+", b.casefold()))
        if not a_tokens or not b_tokens:
            return 0.0
        jaccard = len(a_tokens & b_tokens) / max(len(a_tokens | b_tokens), 1)
        sequence = SequenceMatcher(
            None,
            " ".join(sorted(a_tokens)),
            " ".join(sorted(b_tokens)),
        ).ratio()
        return max(jaccard, sequence)

    def build_ocr_intervals(
        self,
        observations: List[Dict[str, Any]],
        max_gap: float = 1.6,
    ) -> List[Dict[str, Any]]:
        intervals: List[Dict[str, Any]] = []
        for observation in sorted(
            observations,
            key=lambda item: item.get("timestamp", 0.0),
        ):
            text = observation.get("text", "")
            if not text:
                continue
            timestamp = float(observation.get("timestamp", 0.0))
            if (
                intervals
                and timestamp - intervals[-1]["end_ts"] <= max_gap
                and self._similarity(text, intervals[-1]["text"]) >= 0.45
            ):
                intervals[-1]["end_ts"] = timestamp
                intervals[-1]["observations"].append(observation)
                if len(text) > len(intervals[-1]["text"]):
                    intervals[-1]["text"] = text
            else:
                intervals.append({
                    "text": text,
                    "start_ts": timestamp,
                    "end_ts": timestamp,
                    "observations": [observation],
                })

        for interval in intervals:
            if interval["end_ts"] <= interval["start_ts"]:
                interval["end_ts"] = interval["start_ts"] + 0.5
        return intervals

    def _select_frames(
        self,
        keyframes: List[Dict[str, Any]],
        max_frames: Optional[int] = None,
    ) -> List[Dict[str, Any]]:
        limit = max_frames or self.max_frames_per_clip
        if len(keyframes) <= limit:
            return keyframes
        indices = [
            round(index * (len(keyframes) - 1) / (limit - 1))
            for index in range(limit)
        ]
        return [keyframes[index] for index in indices]

    def process_clip_ocr_detailed(
        self,
        keyframes: List[Dict[str, Any]],
        max_frames: Optional[int] = None,
    ) -> Dict[str, Any]:
        self.load_model()
        selected = self._select_frames(keyframes, max_frames=max_frames)

        observations = []
        for frame in selected:
            observations.append(
                self.extract_regions_from_frame(
                    frame.get("path", ""),
                    frame.get("timestamp", 0.0),
                    frame.get("frame_id", ""),
                )
            )

        combined = " ".join(dict.fromkeys(
            observation["text"]
            for observation in observations
            if observation.get("text")
        ))
        return {
            "combined_text": combined,
            "observations": observations,
            "intervals": self.build_ocr_intervals(observations),
            "processed_frames": len(observations),
            "nonempty_frames": sum(
                1 for observation in observations if observation.get("text")
            ),
            "region_count": sum(
                len(observation.get("regions", []))
                for observation in observations
            ),
        }

    def process_clip_ocr(
        self,
        keyframe_paths: List[str],
        max_frames: int = 4,
    ) -> str:
        keyframes = [
            {
                "frame_id": f"frame_{index}",
                "path": path,
                "timestamp": float(index),
            }
            for index, path in enumerate(keyframe_paths)
        ]
        return self.process_clip_ocr_detailed(
            keyframes,
            max_frames=max_frames,
        )["combined_text"]


Writing ingestion/ocr_extractor.py


In [7]:
%%writefile ingestion/embedder.py
"""Lazy BGE-M3 + SigLIP2 embedder pinned to the ingestion GPU."""
import gc
from typing import List, Dict, Any, Union
import numpy as np

try:
    import torch
    from PIL import Image
except Exception:
    torch = None

from search.device_detector import configured_devices, best_dtype, release_cuda


class MultimodalEmbedder:
    TEXT_MODEL_NAME = "BAAI/bge-m3"
    VISUAL_MODEL_NAME = "google/siglip2-base-patch16-224"
    TEXT_DIM = 1024
    VISUAL_DIM = 768

    def __init__(self, device: str = None, autoload: bool = False):
        self.device = device or configured_devices()["ingest"]
        self.dtype = best_dtype(self.device)
        self.sbert_model = None
        self.siglip_model = None
        self.siglip_processor = None
        if autoload:
            self.load_models()

    def load_text_model(self):
        if self.sbert_model is not None:
            return
        from sentence_transformers import SentenceTransformer
        print(f"[MultimodalEmbedder] Loading BGE-M3 on {self.device}...")
        self.sbert_model = SentenceTransformer(
            self.TEXT_MODEL_NAME,
            device=self.device,
        )
        print("[MultimodalEmbedder]  BGE-M3 loaded")

    def load_visual_model(self):
        if self.siglip_model is not None:
            return
        from transformers import AutoProcessor, AutoModel
        print(f"[MultimodalEmbedder] Loading SigLIP2 on {self.device}...")
        self.siglip_processor = AutoProcessor.from_pretrained(
            self.VISUAL_MODEL_NAME,
            use_fast=False,
        )
        try:
            self.siglip_model = AutoModel.from_pretrained(
                self.VISUAL_MODEL_NAME,
                dtype=self.dtype,
            )
        except TypeError:
            self.siglip_model = AutoModel.from_pretrained(
                self.VISUAL_MODEL_NAME,
                torch_dtype=self.dtype,
            )
        self.siglip_model = self.siglip_model.to(
            self.device,
            dtype=self.dtype,
        ).eval()
        print(f"[MultimodalEmbedder]  SigLIP2 loaded ({self.dtype})")

    def load_models(self):
        self.load_text_model()
        self.load_visual_model()
        return self

    def release_models(self):
        for name in ("sbert_model", "siglip_model", "siglip_processor"):
            obj = getattr(self, name, None)
            if obj is not None:
                try:
                    del obj
                except Exception:
                    pass
            setattr(self, name, None)
        gc.collect()
        release_cuda(self.device)
        print(f"[MultimodalEmbedder]  Released models from {self.device}")

    @staticmethod
    def _zero(dim):
        return [0.0] * dim

    @staticmethod
    def _normalize(vector, target_dim):
        vector = np.asarray(vector, dtype=np.float32).reshape(-1)
        if len(vector) > target_dim:
            vector = vector[:target_dim]
        elif len(vector) < target_dim:
            vector = np.pad(vector, (0, target_dim - len(vector)))
        norm = float(np.linalg.norm(vector))
        return (vector / norm).tolist() if norm > 1e-8 else [0.0] * target_dim

    @staticmethod
    def _extract(output):
        if torch is not None and isinstance(output, torch.Tensor):
            return output
        for name in ("image_embeds", "text_embeds", "pooler_output", "last_hidden_state"):
            value = getattr(output, name, None)
            if value is not None:
                return value.mean(dim=1) if name == "last_hidden_state" else value
        return output[0]

    def _move(self, inputs):
        moved = {}
        for key, value in inputs.items():
            if not isinstance(value, torch.Tensor):
                moved[key] = value
            elif torch.is_floating_point(value):
                moved[key] = value.to(self.device, dtype=self.dtype)
            else:
                moved[key] = value.to(self.device)
        return moved

    def embed_speech(self, text):
        if not str(text).strip():
            return self._zero(self.TEXT_DIM)
        self.load_text_model()
        vector = self.sbert_model.encode(
            str(text),
            normalize_embeddings=True,
            show_progress_bar=False,
        )
        return self._normalize(vector, self.TEXT_DIM)

    def embed_ocr(self, text):
        return self.embed_speech(text)

    def embed_query_text(self, text):
        return self.embed_speech(text)

    def embed_visual_frames(self, frames):
        if not frames:
            return []
        self.load_visual_model()
        output = []
        for frame in frames:
            try:
                image = (
                    Image.open(frame).convert("RGB")
                    if isinstance(frame, str)
                    else Image.fromarray(frame)
                )
                inputs = self._move(
                    self.siglip_processor(images=[image], return_tensors="pt")
                )
                with torch.inference_mode():
                    if hasattr(self.siglip_model, "get_image_features"):
                        feature = self.siglip_model.get_image_features(**inputs)
                    else:
                        feature = self._extract(self.siglip_model(**inputs))
                feature = self._extract(feature).float()
                feature = feature / (feature.norm(dim=-1, keepdim=True) + 1e-8)
                output.append(
                    self._normalize(
                        feature.detach().to("cpu", dtype=torch.float32).numpy(),
                        self.VISUAL_DIM,
                    )
                )
            except Exception as exc:
                print(f"[MultimodalEmbedder] Frame embedding failed: {exc}")
                output.append(self._zero(self.VISUAL_DIM))
        return output

    def embed_visual(self, frames):
        vectors = [
            vector for vector in self.embed_visual_frames(frames)
            if np.linalg.norm(vector) > 1e-8
        ]
        if not vectors:
            return self._zero(self.VISUAL_DIM)
        return self._normalize(np.mean(vectors, axis=0), self.VISUAL_DIM)

    def embed_query_visual(self, text):
        if not str(text).strip():
            return self._zero(self.VISUAL_DIM)
        self.load_visual_model()
        try:
            inputs = self._move(
                self.siglip_processor(
                    text=[str(text)],
                    return_tensors="pt",
                    padding="max_length",
                    max_length=64,
                    truncation=True,
                )
            )
            with torch.inference_mode():
                if hasattr(self.siglip_model, "get_text_features"):
                    feature = self.siglip_model.get_text_features(**inputs)
                else:
                    feature = self._extract(self.siglip_model(**inputs))
            feature = self._extract(feature).float()
            feature = feature / (feature.norm(dim=-1, keepdim=True) + 1e-8)
            return self._normalize(
                feature.detach().to("cpu", dtype=torch.float32).numpy(),
                self.VISUAL_DIM,
            )
        except Exception as exc:
            print(f"[MultimodalEmbedder] Visual-query failure: {exc}")
            return self._zero(self.VISUAL_DIM)

    def embed_clips(self, clips: List[Dict[str, Any]]):
        self.load_models()
        for clip in clips:
            keyframes = clip.get("keyframes", [])
            paths = [
                frame.get("path", "")
                for frame in keyframes
                if frame.get("path")
            ]
            visual_vectors = self.embed_visual_frames(paths)
            observations = {
                item.get("frame_id"): item
                for item in clip.get("ocr_observations", [])
            }
            frame_records = []
            for frame, visual_vector in zip(keyframes, visual_vectors):
                observation = observations.get(frame.get("frame_id"), {})
                frame_text = observation.get("text", "")
                frame_records.append({
                    **frame,
                    "visual_embedding": visual_vector,
                    "ocr_text": frame_text,
                    "ocr_embedding": self.embed_ocr(frame_text),
                    "ocr_regions": observation.get("regions", []),
                })
            clip["frame_records"] = frame_records
            clip["visual_embedding"] = self.embed_visual(paths)
            clip["speech_embedding"] = self.embed_speech(
                clip.get("transcript", "")
            )
            clip["ocr_embedding"] = self.embed_ocr(
                clip.get("ocr_text", "")
            )
        return clips


Writing ingestion/embedder.py


In [8]:
%%writefile ingestion/indexer.py
"""Qdrant clip + frame index for coarse-to-fine spatio-temporal retrieval."""
import os
import uuid
import json
from typing import List, Dict, Any, Optional

import numpy as np


class QdrantIndexer:
    COLLECTION_NAME = "spatio_temporal_clips"
    FRAME_COLLECTION_NAME = "spatio_temporal_frames"
    VISUAL_DIM = 768
    TEXT_DIM = 1024

    def __init__(
        self,
        collection_name: str = None,
        location: str = "storage/qdrant_db",
    ):
        self.collection_name = collection_name or self.COLLECTION_NAME
        self.frame_collection_name = self.FRAME_COLLECTION_NAME
        self.client = None
        self.clip_cache_file = os.path.join("storage", f"index_{self.collection_name}.json")
        self.frame_cache_file = os.path.join("storage", f"index_{self.frame_collection_name}.json")
        self.clip_storage = self._load_cache(self.clip_cache_file)
        self.frame_storage = self._load_cache(self.frame_cache_file)
        self.local_storage = self.clip_storage

        try:
            from qdrant_client import QdrantClient

            self.location = location
            if location == ":memory:":
                self.client = QdrantClient(location=":memory:")
            else:
                os.makedirs(location, exist_ok=True)
                self.client = QdrantClient(path=location)

            self._ensure_collections()
        except Exception as exc:
            print(f"[QdrantIndexer] Local fallback active: {exc}")
            self.client = None

    @staticmethod
    def _load_cache(path: str) -> Dict[str, Any]:
        if os.path.exists(path):
            try:
                with open(path, "r", encoding="utf-8") as handle:
                    return json.load(handle)
            except Exception:
                pass
        return {}

    @staticmethod
    def _save_cache(path: str, data: Dict[str, Any]):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        with open(path, "w", encoding="utf-8") as handle:
            json.dump(data, handle, ensure_ascii=False)

    def _save_disk_cache(self):
        self._save_cache(self.clip_cache_file, self.clip_storage)
        self._save_cache(self.frame_cache_file, self.frame_storage)

    def _ensure_collections(self):
        from qdrant_client.models import VectorParams, Distance
        existing = {collection.name for collection in self.client.get_collections().collections}
        if self.collection_name not in existing:
            self.client.create_collection(
                collection_name=self.collection_name,
                vectors_config={
                    "visual": VectorParams(size=self.VISUAL_DIM, distance=Distance.COSINE),
                    "speech": VectorParams(size=self.TEXT_DIM, distance=Distance.COSINE),
                    "ocr": VectorParams(size=self.TEXT_DIM, distance=Distance.COSINE),
                },
            )
        if self.frame_collection_name not in existing:
            self.client.create_collection(
                collection_name=self.frame_collection_name,
                vectors_config={
                    "visual": VectorParams(size=self.VISUAL_DIM, distance=Distance.COSINE),
                    "ocr": VectorParams(size=self.TEXT_DIM, distance=Distance.COSINE),
                },
            )

    @staticmethod
    def _candidate_id(video_id: str, clip_id: str) -> str:
        return f"{video_id}:{clip_id}"

    @staticmethod
    def _frame_candidate_id(video_id: str, clip_id: str, frame_id: str) -> str:
        return f"{video_id}:{clip_id}:{frame_id}"

    @staticmethod
    def _point_id(identity: str) -> str:
        return str(uuid.uuid5(uuid.NAMESPACE_URL, f"video-rag://{identity}"))

    def _build_filter(self, video_id=None, query_filter=None):
        if self.client is None:
            return None
        from qdrant_client.models import Filter, FieldCondition, MatchValue
        conditions = []
        if video_id:
            conditions.append(FieldCondition(key="video_id", match=MatchValue(value=video_id)))
        for key, value in (query_filter or {}).items():
            if isinstance(value, (list, tuple, set)):
                try:
                    from qdrant_client.models import MatchAny
                    conditions.append(FieldCondition(key=key, match=MatchAny(any=list(value))))
                except Exception:
                    continue
            else:
                conditions.append(FieldCondition(key=key, match=MatchValue(value=value)))
        return Filter(must=conditions) if conditions else None

    def _delete_video_records(self, video_id: str):
        if self.client is not None:
            try:
                from qdrant_client.models import Filter, FieldCondition, MatchValue, FilterSelector
                selector = FilterSelector(
                    filter=Filter(must=[FieldCondition(key="video_id", match=MatchValue(value=video_id))])
                )
                for collection in (self.collection_name, self.frame_collection_name):
                    self.client.delete(collection_name=collection, points_selector=selector, wait=True)
            except Exception as exc:
                print(f"[QdrantIndexer] Cleanup notice: {exc}")

        for storage in (self.clip_storage, self.frame_storage):
            stale = [
                key for key, item in storage.items()
                if (item.get("payload", {}) or {}).get("video_id") == video_id
            ]
            for key in stale:
                storage.pop(key, None)

    @staticmethod
    def _keywords(clip: Dict[str, Any]) -> List[str]:
        tokens = set()
        for field in ("transcript", "ocr_text"):
            for token in str(clip.get(field, "")).casefold().split():
                token = token.strip(".,;:!?\"'()[]{}")
                if len(token) > 2:
                    tokens.add(token)
        return sorted(tokens)[:80]

    def index_clips(
        self, video_id: str, clips: List[Dict[str, Any]], source_video_path: str = ""
    ) -> int:
        self._delete_video_records(video_id)
        clip_points, frame_points = [], []

        try:
            from qdrant_client.models import PointStruct
        except Exception:
            PointStruct = None

        for clip in clips:
            clip_id = clip.get("clip_id", "")
            candidate_id = self._candidate_id(video_id, clip_id)
            payload = {
                "candidate_id": candidate_id,
                "clip_id": clip_id,
                "video_id": video_id,
                "source_video_path": source_video_path or clip.get("source_video_path", ""),
                "start_ts": float(clip.get("start_ts", 0.0)),
                "end_ts": float(clip.get("end_ts", 0.0)),
                "duration": float(clip.get("duration", 0.0)),
                "scene_id": clip.get("scene_id", ""),
                "transcript": clip.get("transcript", ""),
                "speech_text": clip.get("transcript", ""),
                "ocr_text": clip.get("ocr_text", ""),
                "keywords": self._keywords(clip),
                "word_timestamps": clip.get("word_timestamps", []),
                "ocr_observations": clip.get("ocr_observations", []),
                "ocr_intervals": clip.get("ocr_intervals", []),
                "keyframes": [
                    {key: value for key, value in frame.items()
                     if key not in {"visual_embedding", "ocr_embedding"}}
                    for frame in clip.get("frame_records", clip.get("keyframes", []))
                ],
                "keyframe_paths": clip.get("keyframe_paths", []),
            }
            vectors = {
                "visual": clip.get("visual_embedding", [0.0] * self.VISUAL_DIM),
                "speech": clip.get("speech_embedding", [0.0] * self.TEXT_DIM),
                "ocr": clip.get("ocr_embedding", [0.0] * self.TEXT_DIM),
            }
            self.clip_storage[candidate_id] = {
                "id": candidate_id,
                "candidate_id": candidate_id,
                "clip_id": clip_id,
                "video_id": video_id,
                "vectors": vectors,
                "payload": payload,
            }
            if PointStruct is not None:
                clip_points.append(PointStruct(
                    id=self._point_id(candidate_id), vector=vectors, payload=payload
                ))

            for frame in clip.get("frame_records", []):
                frame_id = frame.get("frame_id", "")
                frame_candidate_id = self._frame_candidate_id(video_id, clip_id, frame_id)
                frame_payload = {
                    "frame_candidate_id": frame_candidate_id,
                    "candidate_id": candidate_id,
                    "frame_id": frame_id,
                    "clip_id": clip_id,
                    "video_id": video_id,
                    "timestamp": float(frame.get("timestamp", 0.0)),
                    "frame_index": int(frame.get("frame_index", 0)),
                    "path": frame.get("path", ""),
                    "width": int(frame.get("width", 0)),
                    "height": int(frame.get("height", 0)),
                    "ocr_text": frame.get("ocr_text", ""),
                    "ocr_regions": frame.get("ocr_regions", []),
                }
                frame_vectors = {
                    "visual": frame.get("visual_embedding", [0.0] * self.VISUAL_DIM),
                    "ocr": frame.get("ocr_embedding", [0.0] * self.TEXT_DIM),
                }
                self.frame_storage[frame_candidate_id] = {
                    "id": frame_candidate_id,
                    "candidate_id": candidate_id,
                    "frame_id": frame_id,
                    "clip_id": clip_id,
                    "video_id": video_id,
                    "vectors": frame_vectors,
                    "payload": frame_payload,
                }
                if PointStruct is not None:
                    frame_points.append(PointStruct(
                        id=self._point_id(frame_candidate_id),
                        vector=frame_vectors,
                        payload=frame_payload,
                    ))

        if self.client is not None and PointStruct is not None:
            try:
                if clip_points:
                    self.client.upsert(self.collection_name, clip_points, wait=True)
                if frame_points:
                    self.client.upsert(self.frame_collection_name, frame_points, wait=True)
            except Exception as exc:
                print(f"[QdrantIndexer] Qdrant upsert failed; local index retained: {exc}")

        self._save_disk_cache()
        return len(clips)

    @staticmethod
    def _map_qdrant_results(results):
        mapped = []
        for result in results:
            payload = result.payload or {}
            mapped.append({
                "candidate_id": payload.get("candidate_id", ""),
                "frame_candidate_id": payload.get("frame_candidate_id", ""),
                "clip_id": payload.get("clip_id", ""),
                "frame_id": payload.get("frame_id", ""),
                "video_id": payload.get("video_id", ""),
                "payload": payload,
                "score": float(result.score),
            })
        return mapped

    def search(
        self,
        collection_name: str,
        query_vector: tuple,
        limit: int = 10,
        query_filter: Optional[Dict] = None,
        video_id: Optional[str] = None,
    ) -> List[Dict[str, Any]]:
        vector_name, vector = query_vector
        collection = collection_name or self.collection_name
        if self.client is not None:
            try:
                response = self.client.query_points(
                    collection_name=collection,
                    query=vector,
                    using=vector_name,
                    query_filter=self._build_filter(video_id, query_filter),
                    with_payload=True,
                    limit=limit,
                )
                return self._map_qdrant_results(response.points)
            except Exception as exc:
                print(f"[QdrantIndexer] Qdrant search fallback: {exc}")
        storage = self.frame_storage if collection == self.frame_collection_name else self.clip_storage
        return self._local_search(storage, vector_name, vector, video_id, query_filter, limit)

    def search_frames(
        self,
        query_vector: tuple,
        video_id: Optional[str] = None,
        clip_ids: Optional[List[str]] = None,
        limit: int = 20,
    ) -> List[Dict[str, Any]]:
        query_filter = {"clip_id": clip_ids} if clip_ids else None
        results = self.search(
            collection_name=self.frame_collection_name,
            query_vector=query_vector,
            limit=max(limit * 4 if clip_ids else limit, limit),
            query_filter=query_filter,
            video_id=video_id,
        )
        if clip_ids:
            allowed = set(clip_ids)
            results = [item for item in results if item.get("clip_id") in allowed]
        return results[:limit]

    @staticmethod
    def _local_search(storage, vector_name, vector, video_id, query_filter, limit):
        query = np.asarray(vector, dtype=np.float32)
        query_norm = np.linalg.norm(query)
        if query_norm < 1e-8:
            return []
        hits = []
        for item in storage.values():
            payload = item.get("payload", {})
            if video_id and payload.get("video_id") != video_id:
                continue
            valid = True
            for key, expected in (query_filter or {}).items():
                actual = payload.get(key)
                valid = actual in expected if isinstance(expected, (list, tuple, set)) else actual == expected
                if not valid:
                    break
            if not valid:
                continue
            stored = np.asarray(item.get("vectors", {}).get(vector_name, []), dtype=np.float32)
            if stored.shape != query.shape:
                continue
            norm = np.linalg.norm(stored)
            if norm < 1e-8:
                continue
            score = float(np.dot(query, stored) / (query_norm * norm))
            hits.append({
                "candidate_id": item.get("candidate_id", payload.get("candidate_id", "")),
                "frame_candidate_id": item.get("id", payload.get("frame_candidate_id", "")),
                "clip_id": item.get("clip_id", payload.get("clip_id", "")),
                "frame_id": item.get("frame_id", payload.get("frame_id", "")),
                "video_id": item.get("video_id", payload.get("video_id", "")),
                "payload": payload,
                "score": score,
            })
        hits.sort(key=lambda item: item["score"], reverse=True)
        return hits[:limit]

    def indexed_count(self, video_id: Optional[str] = None) -> int:
        return sum(
            1 for item in self.clip_storage.values()
            if not video_id or (item.get("payload", {}) or {}).get("video_id") == video_id
        )

    def indexed_frame_count(self, video_id: Optional[str] = None) -> int:
        return sum(
            1 for item in self.frame_storage.values()
            if not video_id or (item.get("payload", {}) or {}).get("video_id") == video_id
        )

    def index_status(self, video_id: Optional[str] = None) -> Dict[str, Any]:
        clips = self.indexed_count(video_id)
        frames = self.indexed_frame_count(video_id)
        return {
            "clip_collection": self.collection_name,
            "frame_collection": self.frame_collection_name,
            "video_id": video_id or "*",
            "indexed_clips": clips,
            "indexed_frames": frames,
            "qdrant_connected": self.client is not None,
            "ready": clips > 0 and frames > 0,
        }

    def clear_collection(self):
        if self.client is not None:
            try:
                self.client.delete_collection(self.collection_name)
                self.client.delete_collection(self.frame_collection_name)
                self._ensure_collections()
            except Exception:
                pass
        self.clip_storage.clear()
        self.frame_storage.clear()
        self._save_disk_cache()


Writing ingestion/indexer.py


In [9]:
%%writefile search/device_detector.py
"""Explicit CUDA device selection for staged dual-GPU execution."""
import gc
import os
from typing import Dict, Any, Tuple

try:
    import torch
except Exception:
    torch = None


def cuda_device(index: int = 0) -> str:
    if torch is None or not torch.cuda.is_available():
        return "cpu"
    count = torch.cuda.device_count()
    safe_index = max(0, min(int(index), count - 1))
    return f"cuda:{safe_index}"


def best_dtype(device: str):
    if torch is None:
        return None
    if not str(device).startswith("cuda"):
        return torch.float32
    index = int(str(device).split(":")[-1])
    major, _ = torch.cuda.get_device_capability(index)
    if major >= 8 and torch.cuda.is_bf16_supported():
        return torch.bfloat16
    return torch.float16


def configured_devices() -> Dict[str, str]:
    """GPU 0 is reserved for embeddings/ingestion; GPU 1 for search models."""
    count = torch.cuda.device_count() if torch is not None and torch.cuda.is_available() else 0
    ingest_index = int(os.environ.get("VIDEO_RAG_INGEST_GPU", "0"))
    search_default = "1" if count >= 2 else "0"
    search_index = int(os.environ.get("VIDEO_RAG_SEARCH_GPU", search_default))
    return {
        "ingest": cuda_device(ingest_index),
        "search": cuda_device(search_index),
        # EasyOCR DataParallel requires cuda:0. It runs after Whisper is
        # released and before BGE/SigLIP are loaded.
        "ocr": os.environ.get(
            "VIDEO_RAG_OCR_DEVICE",
            cuda_device(ingest_index),
        ),
    }


def release_cuda(device: str = None):
    gc.collect()
    if torch is None or not torch.cuda.is_available():
        return
    try:
        if device and str(device).startswith("cuda"):
            with torch.cuda.device(int(str(device).split(":")[-1])):
                torch.cuda.synchronize()
                torch.cuda.empty_cache()
                torch.cuda.ipc_collect()
        else:
            for index in range(torch.cuda.device_count()):
                with torch.cuda.device(index):
                    torch.cuda.synchronize()
                    torch.cuda.empty_cache()
                    torch.cuda.ipc_collect()
    except Exception:
        torch.cuda.empty_cache()


def device_diagnostics() -> Dict[str, Any]:
    devices = configured_devices()
    result = {
        "cuda_available": bool(torch is not None and torch.cuda.is_available()),
        "gpu_count": int(torch.cuda.device_count()) if torch is not None and torch.cuda.is_available() else 0,
        "assignment": devices,
        "gpus": [],
    }
    if result["cuda_available"]:
        for index in range(torch.cuda.device_count()):
            free, total = torch.cuda.mem_get_info(index)
            result["gpus"].append({
                "index": index,
                "name": torch.cuda.get_device_name(index),
                "capability": torch.cuda.get_device_capability(index),
                "dtype": str(best_dtype(f"cuda:{index}")),
                "free_gb": round(free / (1024 ** 3), 2),
                "total_gb": round(total / (1024 ** 3), 2),
            })
    return result


def get_optimal_device() -> Tuple[str, Any]:
    """Backward-compatible helper: returns the ingestion device."""
    device = configured_devices()["ingest"]
    return device, best_dtype(device)


Writing search/device_detector.py


In [10]:
%%writefile search/query_understanding.py
import gc, json, os, re
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any
try:
    import torch
except Exception:
    torch = None
from search.device_detector import configured_devices, release_cuda

VALID_MODALITIES = {"visual", "speech", "ocr"}

@dataclass
class QueryIntent:
    original_query: str
    intent: str = "content_search"
    entities: List[str] = field(default_factory=list)
    temporal_selector: str = "any"
    temporal_relation: str = "none"
    visual_query: str = ""
    speech_query: str = ""
    ocr_query: str = ""
    primary_modality: str = "visual"
    relation_left_modality: str = "visual"
    relation_right_modality: str = "speech"
    needs_visual: bool = True
    needs_speech: bool = True
    needs_ocr: bool = False
    requires_spatial_grounding: bool = True
    modality_weights: Dict[str, float] = field(default_factory=dict)
    expanded_queries: List[str] = field(default_factory=list)

class FastQueryUnderstander:
    MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

    def __init__(self, device: Optional[str] = None, autoload: bool = False, use_llm: Optional[bool] = None):
        self.device = device or configured_devices()["search"]
        self.use_llm = bool(use_llm) if use_llm is not None else os.environ.get("VIDEO_RAG_USE_LLM_PLANNER", "0") == "1"
        self.model = None
        self.tokenizer = None
        if autoload and self.use_llm:
            self.load_model()

    def load_model(self):
        if not self.use_llm or self.model is not None or torch is None or not self.device.startswith("cuda"):
            return self
        try:
            from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
            self.tokenizer = AutoTokenizer.from_pretrained(self.MODEL_NAME)
            q = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                                      bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True)
            self.model = AutoModelForCausalLM.from_pretrained(
                self.MODEL_NAME, quantization_config=q, device_map={"": self.device},
                low_cpu_mem_usage=True).eval()
        except Exception as exc:
            print(f"[QueryUnderstander] deterministic fallback: {exc}")
            self.model = None
        return self

    def release_model(self):
        for name in ("model", "tokenizer"):
            obj = getattr(self, name, None)
            if obj is not None:
                try: del obj
                except Exception: pass
            setattr(self, name, None)
        gc.collect()
        release_cuda(self.device)

    @staticmethod
    def _contains(text, terms):
        t = text.casefold()
        return any(x in t for x in terms)

    @staticmethod
    def _relation(value, query):
        text = f"{value or ''} {query}".casefold()
        if any(x in text for x in ("while", "during", "at the same time", "بينما", "أثناء")): return "while"
        if any(x in text for x in ("before", "قبل")): return "before"
        if any(x in text for x in ("after", "بعد")): return "after"
        return "none"

    @staticmethod
    def _selector(value, query):
        text = f"{value or ''} {query}".casefold()
        if any(x in text for x in ("first", "earliest", "أول")): return "first"
        if any(x in text for x in ("last", "latest", "آخر")): return "last"
        return "any"

    @staticmethod
    def _modality(value, default):
        text = str(value or "").casefold()
        if "speech" in text or "spoken" in text: return "speech"
        if "ocr" in text or "screen text" in text: return "ocr"
        if "visual" in text or "image" in text: return "visual"
        return default

    @staticmethod
    def _weights(raw, active):
        vals = {}
        for m in active:
            try: vals[m] = max(float((raw or {}).get(m, 0)), 0)
            except Exception: vals[m] = 0
        if sum(vals.values()) <= 0: vals = {m: 1 for m in active}
        total = sum(vals.values())
        return {k: v / total for k, v in vals.items()}

    def analyze(self, query):
        query = str(query).strip()
        deterministic = self._fallback(query)
        if not self.use_llm:
            print("[QueryUnderstander] deterministic strict parser")
            return deterministic
        self.load_model()
        if self.model is None: return deterministic
        try:
            data = self._llm(query)
            return self._repair_llm(query, data, deterministic)
        except Exception as exc:
            print(f"[QueryUnderstander] invalid LLM output, fallback: {exc}")
            return deterministic

    def _llm(self, query):
        system = ("Return ONLY JSON. Allowed temporal_selector: first,last,any. "
                  "Allowed temporal_relation: while,before,after,none. "
                  "Allowed modalities: visual,speech,ocr. Include visual_query,speech_query,ocr_query, "
                  "needs_visual,needs_speech,needs_ocr,requires_spatial_grounding,modality_weights.")
        messages = [{"role":"system","content":system},{"role":"user","content":query}]
        prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.device)
        with torch.inference_mode():
            out = self.model.generate(**inputs, max_new_tokens=260, do_sample=False)
        text = self.tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        a, b = text.find("{"), text.rfind("}")
        if a < 0 or b <= a: raise ValueError("no JSON")
        return json.loads(text[a:b+1])

    def _repair_llm(self, query, data, base):
        relation = self._relation(data.get("temporal_relation"), query)
        selector = self._selector(data.get("temporal_selector"), query)
        nv = base.needs_visual or bool(data.get("needs_visual", False))
        ns = base.needs_speech or bool(data.get("needs_speech", False))
        no = base.needs_ocr or bool(data.get("needs_ocr", False))
        active = [m for m, ok in (("visual",nv),("speech",ns),("ocr",no)) if ok] or ["visual","speech","ocr"]
        return QueryIntent(
            original_query=query,
            intent=(
                "temporal_search"
                if relation != "none" or selector != "any"
                else "content_search"
            ),
            entities=list(data.get("entities") or base.entities),
            temporal_selector=selector,
            temporal_relation=relation,
            visual_query=str(data.get("visual_query") or base.visual_query),
            speech_query=str(data.get("speech_query") or base.speech_query),
            ocr_query=str(data.get("ocr_query") or base.ocr_query),
            primary_modality=self._modality(data.get("primary_modality"), base.primary_modality),
            relation_left_modality=self._modality(data.get("relation_left_modality"), base.relation_left_modality),
            relation_right_modality=self._modality(data.get("relation_right_modality"), base.relation_right_modality),
            needs_visual=nv, needs_speech=ns, needs_ocr=no,
            requires_spatial_grounding=base.requires_spatial_grounding or bool(data.get("requires_spatial_grounding", False)),
            modality_weights=self._weights(data.get("modality_weights"), active),
            expanded_queries=list(dict.fromkeys([query] + list(data.get("expanded_queries") or [])))[:3],
        )

    def _fallback(self, query):
        q = f" {query.casefold()} "
        relation = self._relation("", query)
        selector = self._selector("", query)
        ocr_terms = ["written","text","code","command","command-line","terminal","dockerfile","screen","port","expose",
                     "مكتوب","كود","أمر","شاشة","سطر","منفذ"]
        speech_terms = [
            "say", "says", "said",
            "mention", "mentions", "mentioned",
            "speaker", "speaks", "spoken",
            "explain", "explains", "explained",
            "discuss", "discusses", "narrate", "describes",
            "شرح", "يشرح", "يتحدث", "يقول",
        ]
        visual_terms = ["show","shows","appear","display","displays","interface","terminal","screen","person","object",
                        "واجهة","يظهر","يعرض"]
        no = self._contains(q, ocr_terms)
        ns = self._contains(q, speech_terms) or relation != "none"
        nv = self._contains(q, visual_terms) or no or relation != "none"

        connector = next((x.strip() for x in (" while "," during "," at the same time "," بينما "," أثناء ",
                                               " before "," قبل "," after "," بعد ") if x in q), None)
        left, right = query, ""
        if connector:
            parts = re.split(re.escape(connector), query, maxsplit=1, flags=re.IGNORECASE)
            if len(parts) == 2: left, right = parts[0].strip(), parts[1].strip()

        def mod(clause):
            c = clause.casefold()
            if self._contains(c, speech_terms): return "speech"
            if self._contains(c, ocr_terms): return "ocr"
            return "visual"

        lm = mod(left)
        rm = mod(right) if right else ("speech" if lm != "speech" else "visual")
        clauses = {
            "visual": "",
            "speech": "",
            "ocr": "",
        }
        clauses[lm] = left

        if right:
            if rm == lm:
                if self._contains(right, speech_terms):
                    rm = "speech"
                elif self._contains(right, ocr_terms):
                    rm = "ocr"
                else:
                    rm = "visual"
            clauses[rm] = right

        if no and not clauses["ocr"]:
            clauses["ocr"] = left if relation != "none" else query
        if nv and not clauses["visual"]:
            clauses["visual"] = left if relation != "none" else query
        if ns and not clauses["speech"]:
            clauses["speech"] = right or query

        active = [m for m, ok in (("visual",nv),("speech",ns),("ocr",no)) if ok] or ["visual","speech","ocr"]
        if nv and ns and no: weights = {"visual":0.30,"speech":0.30,"ocr":0.40}
        elif nv and no: weights = {"visual":0.35,"ocr":0.65}
        else: weights = {m:1/len(active) for m in active}

        return QueryIntent(
            original_query=query,
            intent="temporal_search" if relation != "none" or selector != "any" else "content_search",
            entities=[x for x in re.findall(r"[\w.-]+", query) if len(x)>2][:16],
            temporal_selector=selector, temporal_relation=relation,
            visual_query=clauses["visual"], speech_query=clauses["speech"], ocr_query=clauses["ocr"],
            primary_modality=lm, relation_left_modality=lm, relation_right_modality=rm,
            needs_visual=nv, needs_speech=ns, needs_ocr=no,
            requires_spatial_grounding=nv or no,
            modality_weights=weights, expanded_queries=[query],
        )


Writing search/query_understanding.py


In [11]:
%%writefile search/retrieval_planner.py
from dataclasses import dataclass, field
from typing import List, Dict, Any

@dataclass
class RetrievalPlan:
    queries: List[str]
    modalities: List[str]
    modality_weights: Dict[str, float]
    queries_by_modality: Dict[str, List[str]] = field(default_factory=dict)
    fusion: str = "rrf"
    rerank: bool = True
    verify: bool = True
    temporal_selector: str = "any"
    temporal_relation: str = "none"
    primary_modality: str = "visual"
    relation_left_modality: str = "visual"
    relation_right_modality: str = "speech"
    requires_spatial_grounding: bool = True
    original_query: str = ""

    @property
    def active_modalities(self): return self.modalities
    @property
    def temporal_intent(self): return self.temporal_selector
    def query_for(self, modality):
        values = self.queries_by_modality.get(modality) or self.queries
        return values[0] if values else self.original_query

class RetrievalPlanner:
    def plan(self, intent: Any, rewritten_queries: List[str]):
        valid_m = {"visual","speech","ocr"}
        relation = intent.temporal_relation if intent.temporal_relation in {"while","before","after","none"} else "none"
        selector = intent.temporal_selector if intent.temporal_selector in {"first","last","any"} else "any"
        left = intent.relation_left_modality if intent.relation_left_modality in valid_m else "visual"
        right = intent.relation_right_modality if intent.relation_right_modality in valid_m else "speech"
        modalities = [m for m, ok in (("visual",intent.needs_visual),("speech",intent.needs_speech),("ocr",intent.needs_ocr)) if ok]
        if relation != "none":
            for m in (left, right):
                if m not in modalities: modalities.append(m)
        if not modalities: modalities = ["visual","speech","ocr"]

        raw = intent.modality_weights or {}
        weights = {}
        for m in modalities:
            try: weights[m] = max(float(raw.get(m,0)),0)
            except Exception: weights[m] = 0
        if sum(weights.values()) <= 0: weights = {m:1 for m in modalities}
        total = sum(weights.values())
        weights = {k:v/total for k,v in weights.items()}

        qbm = {}
        for m, attr in (("visual","visual_query"),("speech","speech_query"),("ocr","ocr_query")):
            clause = str(getattr(intent, attr, "") or "").strip()
            qbm[m] = list(dict.fromkeys([x for x in ([clause] if clause else []) + list(rewritten_queries) if x]))[:3]

        return RetrievalPlan(
            queries=list(dict.fromkeys(rewritten_queries)), modalities=modalities,
            modality_weights=weights, queries_by_modality=qbm,
            fusion="rrf" if len(modalities)>1 else "single",
            temporal_selector=selector, temporal_relation=relation,
            primary_modality=intent.primary_modality if intent.primary_modality in valid_m else modalities[0],
            relation_left_modality=left, relation_right_modality=right,
            requires_spatial_grounding=bool(intent.requires_spatial_grounding),
            original_query=intent.original_query,
        )


Writing search/retrieval_planner.py


In [12]:
%%writefile search/temporal_reasoner.py
class TemporalReasoner:
    @staticmethod
    def _start(c):
        return float(c.get("refined",{}).get("refined_start", c.get("payload",{}).get("start_ts",0)))

    def select(self, candidates, selector="any"):
        if not candidates: return []
        relation_required = any(
            c.get("refined",{}).get("temporal_relation","none") in {"while","before","after"}
            for c in candidates
        )
        valid = [c for c in candidates if c.get("refined",{}).get("relation_satisfied",True)]
        if relation_required and not valid: return []
        pool = valid if relation_required else candidates
        if selector == "first": return sorted(pool, key=self._start)
        if selector == "last": return sorted(pool, key=self._start, reverse=True)
        return sorted(pool, key=lambda c:c.get("final_confidence",0), reverse=True)

    def apply(self, candidates, temporal_intent="any"):
        return self.select(candidates, temporal_intent)


Writing search/temporal_reasoner.py


In [13]:
%%writefile search/query_planner.py
"""Query planner whose Qwen model is loaded only for the planning stage."""
from typing import Optional
from .query_understanding import FastQueryUnderstander
from .retrieval_planner import RetrievalPlanner


class QueryPlanner:
    def __init__(self, device: Optional[str] = None):
        self.understander = FastQueryUnderstander(
            device=device,
            autoload=False,
        )
        self.retrieval_planner = RetrievalPlanner()

    def plan(self, query):
        intent = self.understander.analyze(query)
        active = [m for m, ok in (
            ("visual", intent.needs_visual),
            ("speech", intent.needs_speech),
            ("ocr", intent.needs_ocr),
        ) if ok]
        print(
            f"[QueryPlanner] intent={intent.intent} "
            f"selector={intent.temporal_selector} "
            f"relation={intent.temporal_relation} "
            f"modalities={active}"
        )
        print(
            f"[QueryPlanner] visual={intent.visual_query!r} "
            f"speech={intent.speech_query!r} "
            f"ocr={intent.ocr_query!r}"
        )
        return self.retrieval_planner.plan(
            intent,
            intent.expanded_queries,
        )

    def release_model(self):
        self.understander.release_model()


Writing search/query_planner.py


In [14]:
%%writefile search/retriever.py
"""Coarse clip retrieval plus fine frame retrieval and weighted RRF fusion."""
from collections import defaultdict
from typing import List, Dict, Any, Optional


class MultimodalRetriever:
    def __init__(self, indexer: Any, embedder: Any):
        self.indexer = indexer
        self.embedder = embedder

    @staticmethod
    def _identity(result):
        payload = result.get("payload", {}) or {}
        video_id = result.get("video_id") or payload.get("video_id", "")
        clip_id = result.get("clip_id") or payload.get("clip_id", "")
        return f"{video_id}:{clip_id}" if video_id and clip_id else result.get("candidate_id", "")

    def _query_vector(self, modality: str, query: str):
        return (
            self.embedder.embed_query_visual(query)
            if modality == "visual"
            else self.embedder.embed_query_text(query)
        )

    @staticmethod
    def _normalize_ocr_text(text):
        import re
        return " ".join(
            re.findall(r"[a-z0-9._:/-]+", str(text).casefold())
        )

    @classmethod
    def _ocr_lexical_score(cls, query, text):
        import re
        query_norm = cls._normalize_ocr_text(query)
        text_norm = cls._normalize_ocr_text(text)
        if not query_norm or not text_norm:
            return 0.0

        code_targets = []
        for match in re.finditer(
            r"\b[A-Z][A-Z0-9_-]{2,}(?:\s+[:=/-]?\s*\d+(?:[.:/-]\d+)*)?",
            str(query),
        ):
            target = cls._normalize_ocr_text(match.group(0))
            if target:
                code_targets.append(target)

        for target in code_targets:
            if target in text_norm:
                return 1.0

        numbers = set(re.findall(r"\b\d{2,}\b", query_norm))
        if numbers and numbers.issubset(set(text_norm.split())):
            return 0.85

        q_tokens = set(query_norm.split())
        t_tokens = set(text_norm.split())
        return len(q_tokens & t_tokens) / max(len(q_tokens), 1)

    def search(self, plan, video_id: Optional[str] = None, top_k: int = 20):
        hits_per_modality = {}
        for modality in plan.modalities:
            merged = {}
            for query in plan.queries_by_modality.get(modality, plan.queries):
                vector = self._query_vector(modality, query)
                hits = self.indexer.search(
                    collection_name=self.indexer.collection_name,
                    query_vector=(modality, vector),
                    limit=top_k * 3,
                    video_id=video_id,
                )
                for hit in hits:
                    identity = self._identity(hit)
                    if identity and (
                        identity not in merged or hit.get("score", 0.0) > merged[identity].get("score", 0.0)
                    ):
                        merged[identity] = hit
            hits_per_modality[modality] = sorted(
                merged.values(), key=lambda item: item.get("score", 0.0), reverse=True
            )[:top_k * 2]

        print("[Retriever] coarse hits:", {name: len(items) for name, items in hits_per_modality.items()})
        active = {name: items for name, items in hits_per_modality.items() if items}
        if not active:
            return []
        candidates = self.weighted_rrf(
            active,
            plan.modality_weights,
        )

        if "ocr" in plan.modalities:
            ocr_query = plan.query_for("ocr")
            for candidate in candidates:
                payload = candidate.get("payload", {}) or {}
                lexical_score = self._ocr_lexical_score(
                    ocr_query,
                    payload.get("ocr_text", ""),
                )
                candidate["ocr_lexical_score"] = lexical_score
                candidate["retrieval_similarity"] = min(
                    1.0,
                    0.75 * candidate.get(
                        "retrieval_similarity", 0.0
                    )
                    + 0.25 * lexical_score,
                )

            candidates.sort(
                key=lambda item: (
                    item.get("ocr_lexical_score", 0.0),
                    item.get("rrf_score", 0.0),
                ),
                reverse=True,
            )

        return candidates[:top_k]

    def weighted_rrf(self, hits_per_modality, weights, k: int = 60):
        aggregated = {}
        for modality, hits in hits_per_modality.items():
            weight = float(weights.get(modality, 1.0))
            for rank, hit in enumerate(hits):
                identity = self._identity(hit)
                payload = dict(hit.get("payload", {}) or {})
                payload["candidate_id"] = identity
                if identity not in aggregated:
                    aggregated[identity] = {
                        "candidate_id": identity,
                        "clip_id": hit.get("clip_id", payload.get("clip_id", "")),
                        "video_id": hit.get("video_id", payload.get("video_id", "")),
                        "payload": payload,
                        "rrf_score": 0.0,
                        "per_modality_scores": {},
                    }
                item = aggregated[identity]
                item["rrf_score"] += weight / (k + rank + 1)
                item["per_modality_scores"][modality] = max(
                    hit.get("score", 0.0),
                    item["per_modality_scores"].get(modality, -1.0),
                )
                for key, value in payload.items():
                    if value not in (None, "", [], {}):
                        item["payload"][key] = value

        for item in aggregated.values():
            scores = item["per_modality_scores"]
            item["retrieval_similarity"] = sum(
                float(weights.get(name, 0.0)) * max(min(float(score), 1.0), 0.0)
                for name, score in scores.items()
            )
        return sorted(aggregated.values(), key=lambda item: item["rrf_score"], reverse=True)

    def attach_fine_frame_evidence(
        self, plan, candidates: List[Dict[str, Any]], video_id: Optional[str] = None, top_frames: int = 16
    ):
        if not candidates:
            return candidates
        top_clip_ids = [item.get("clip_id", "") for item in candidates[:10]]
        evidence_by_candidate = defaultdict(list)

        for modality in ("visual", "ocr"):
            if modality not in plan.modalities:
                continue
            query = plan.query_for(modality)
            vector = self._query_vector(modality, query)
            hits = self.indexer.search_frames(
                query_vector=(modality, vector),
                video_id=video_id,
                clip_ids=top_clip_ids,
                limit=top_frames,
            )
            for hit in hits:
                payload = hit.get("payload", {}) or {}
                candidate_id = payload.get("candidate_id") or self._identity(hit)
                evidence_by_candidate[candidate_id].append({
                    "modality": modality,
                    "score": float(hit.get("score", 0.0)),
                    "timestamp": float(payload.get("timestamp", 0.0)),
                    "path": payload.get("path", ""),
                    "frame_id": payload.get("frame_id", ""),
                    "width": int(payload.get("width", 0)),
                    "height": int(payload.get("height", 0)),
                    "ocr_text": payload.get("ocr_text", ""),
                    "ocr_regions": payload.get("ocr_regions", []),
                })

        for candidate in candidates:
            evidence = sorted(
                evidence_by_candidate.get(candidate["candidate_id"], []),
                key=lambda item: item["score"],
                reverse=True,
            )
            candidate["frame_evidence"] = evidence
            candidate["payload"]["frame_evidence"] = evidence
            if evidence:
                visual_scores = [item["score"] for item in evidence if item["modality"] == "visual"]
                ocr_scores = [item["score"] for item in evidence if item["modality"] == "ocr"]
                candidate["fine_visual_score"] = max(visual_scores, default=0.0)
                candidate["fine_ocr_score"] = max(ocr_scores, default=0.0)
                candidate["retrieval_similarity"] = min(
                    1.0,
                    0.75 * candidate.get("retrieval_similarity", 0.0)
                    + 0.15 * max(candidate["fine_visual_score"], 0.0)
                    + 0.10 * max(candidate["fine_ocr_score"], 0.0),
                )
        return candidates


Writing search/retriever.py


In [15]:
%%writefile search/reranker.py
"""Lazy BGE reranker pinned to the search GPU."""
import gc
from typing import List, Dict, Any, Optional

try:
    import torch
except Exception:
    torch = None

from search.device_detector import configured_devices, release_cuda


class CrossEncoderReranker:
    MODEL_NAME = "BAAI/bge-reranker-v2-m3"

    def __init__(self, device: Optional[str] = None, autoload: bool = False):
        self.device = device or configured_devices()["search"]
        self.model = None
        self.tokenizer = None
        if autoload:
            self.load_model()

    def load_model(self):
        if self.model is not None:
            return self
        try:
            from transformers import (
                AutoTokenizer,
                AutoModelForSequenceClassification,
            )
            print(f"[Reranker] Loading BGE reranker on {self.device}...")
            self.tokenizer = AutoTokenizer.from_pretrained(self.MODEL_NAME)
            kwargs = {"low_cpu_mem_usage": True}
            if torch is not None and self.device.startswith("cuda"):
                kwargs["torch_dtype"] = torch.float16
            self.model = AutoModelForSequenceClassification.from_pretrained(
                self.MODEL_NAME,
                **kwargs,
            ).to(self.device)
            self.model.eval()
            print("[Reranker]  Loaded")
        except Exception as exc:
            print(f"[Reranker] Retrieval-score fallback: {exc}")
            self.model = None
        return self

    def release_model(self):
        for name in ("model", "tokenizer"):
            obj = getattr(self, name, None)
            if obj is not None:
                try:
                    del obj
                except Exception:
                    pass
            setattr(self, name, None)
        gc.collect()
        release_cuda(self.device)
        print(f"[Reranker]  Released from {self.device}")

    @staticmethod
    def _text(candidate):
        payload = candidate.get("payload", {}) or {}
        parts = [
            payload.get("transcript", ""),
            payload.get("ocr_text", ""),
        ]
        return "\n".join(part for part in parts if part).strip()

    def rerank(self, query: str, candidates: List[Dict[str, Any]], top_k: int = 30):
        if not candidates:
            return candidates
        self.load_model()

        if self.model is None or self.tokenizer is None:
            for candidate in candidates:
                candidate["cross_score"] = max(
                    0.0,
                    min(
                        float(candidate.get("retrieval_similarity", 0.0)),
                        1.0,
                    ),
                )
            return sorted(
                candidates,
                key=lambda item: item["cross_score"],
                reverse=True,
            )

        selected = candidates[:top_k]
        texts = [self._text(candidate) for candidate in selected]
        pairs = [[query, text] for text in texts]
        inputs = self.tokenizer(
            pairs,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt",
        ).to(self.device)

        with torch.inference_mode():
            logits = self.model(**inputs).logits
        if logits.ndim == 2 and logits.shape[-1] > 1:
            scores = torch.softmax(logits.float(), dim=-1)[:, -1]
        else:
            scores = torch.sigmoid(logits.float().reshape(-1))

        for candidate, score in zip(
            selected,
            scores.detach().cpu().tolist(),
        ):
            model_score = float(score)
            lexical_score = float(
                candidate.get("ocr_lexical_score", 0.0)
            )
            candidate["cross_model_score"] = model_score
            candidate["cross_score"] = min(
                1.0,
                0.80 * model_score + 0.20 * lexical_score,
            )
        for candidate in candidates[top_k:]:
            candidate["cross_score"] = 0.0
        return sorted(
            candidates,
            key=lambda item: item.get("cross_score", 0.0),
            reverse=True,
        )


Writing search/reranker.py


In [16]:
%%writefile search/verification.py
"""Lazy Qwen2.5-VL grounding pinned to the search GPU."""
import gc
import json
import os
import re
from difflib import SequenceMatcher
from typing import List, Dict, Any, Optional

try:
    import torch
    from PIL import Image
except Exception:
    torch = None

from search.device_detector import configured_devices, release_cuda


class VideoVerificationLayer:
    MODEL_NAME = "Qwen/Qwen2.5-VL-3B-Instruct"

    def __init__(
        self,
        enable: bool = True,
        device: Optional[str] = None,
        autoload: bool = False,
    ):
        self.enable = bool(enable)
        self.device = device or configured_devices()["search"]
        self.model = None
        self.processor = None
        if autoload and self.enable:
            self.load_model()

    def load_model(self):
        if not self.enable or self.model is not None:
            return self
        if torch is None or not self.device.startswith("cuda"):
            return self
        try:
            from transformers import (
                Qwen2_5_VLForConditionalGeneration,
                AutoProcessor,
                BitsAndBytesConfig,
            )
            quantization = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type="nf4",
            )
            print(f"[VerificationLayer] Loading Qwen2.5-VL on {self.device}...")
            self.model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
                self.MODEL_NAME,
                quantization_config=quantization,
                device_map={"": self.device},
                low_cpu_mem_usage=True,
            )
            self.model.eval()
            self.processor = AutoProcessor.from_pretrained(
                self.MODEL_NAME,
                use_fast=False,
            )
            print("[VerificationLayer]  Loaded")
        except Exception as exc:
            print(f"[VerificationLayer] Indexed-evidence fallback: {exc}")
            self.model = None
        return self

    def release_model(self):
        for name in ("model", "processor"):
            obj = getattr(self, name, None)
            if obj is not None:
                try:
                    del obj
                except Exception:
                    pass
            setattr(self, name, None)
        gc.collect()
        release_cuda(self.device)
        print(f"[VerificationLayer]  Released from {self.device}")

    @staticmethod
    def _normalize_text(text: str) -> str:
        normalized = str(text).casefold()
        normalized = re.sub(r"[_:/=]+", " ", normalized)
        normalized = re.sub(r"[-]+", " ", normalized)
        normalized = " ".join(
            re.findall(r"[a-z0-9.]+", normalized)
        )
        return normalized.strip()

    @classmethod
    def _extract_ocr_targets(cls, query: str):
        """Return OCR targets ordered by semantic specificity.

        Examples:
        - EXPOSE 8080 -> priority 100
        - Dockerfile -> priority 60
        - 8080 -> priority 20

        Lower-priority numeric targets are never allowed to override a richer
        command-plus-number target present in the same query.
        """
        raw_query = str(query)
        targets = []

        # Highest priority: command/identifier followed by a number.
        for match in re.finditer(
            r"\b([A-Z][A-Z0-9_-]{2,})\s+([:=/-]?\s*\d+(?:[.:/-]\d+)*)",
            raw_query,
        ):
            text = cls._normalize_text(
                f"{match.group(1)} {match.group(2)}"
            )
            if text:
                targets.append({
                    "text": text,
                    "priority": 100,
                    "kind": "command_number",
                })

        # Quoted code/text phrases.
        for match in re.finditer(
            r'["\']([^"\']{2,})["\']',
            raw_query,
        ):
            text = cls._normalize_text(match.group(1))
            if text:
                targets.append({
                    "text": text,
                    "priority": 80,
                    "kind": "quoted_phrase",
                })

        # File names and common technical identifiers.
        for match in re.finditer(
            r"\b(?:Dockerfile|Makefile|[A-Za-z0-9_-]+\.(?:py|js|ts|json|yaml|yml|sh|txt|md|html|css))\b",
            raw_query,
            flags=re.IGNORECASE,
        ):
            text = cls._normalize_text(match.group(0))
            if text:
                targets.append({
                    "text": text,
                    "priority": 60,
                    "kind": "filename",
                })

        # Bare uppercase identifiers, but avoid duplicating richer targets.
        for match in re.finditer(
            r"\b[A-Z][A-Z0-9_-]{2,}\b",
            raw_query,
        ):
            text = cls._normalize_text(match.group(0))
            if text:
                targets.append({
                    "text": text,
                    "priority": 50,
                    "kind": "identifier",
                })

        # Lowest priority: standalone numbers.
        for number in re.findall(r"\b\d{2,}\b", raw_query):
            targets.append({
                "text": cls._normalize_text(number),
                "priority": 20,
                "kind": "number",
            })

        # Deduplicate while preserving the highest priority.
        best = {}
        for target in targets:
            text = target["text"]
            if (
                text not in best
                or target["priority"] > best[text]["priority"]
            ):
                best[text] = target

        return sorted(
            best.values(),
            key=lambda item: (
                item["priority"],
                len(item["text"]),
            ),
            reverse=True,
        )

    @classmethod
    def _match_ocr_region(cls, query: str, region_text: str):
        query_norm = cls._normalize_text(query)
        region_norm = cls._normalize_text(region_text)
        if not query_norm or not region_norm:
            return {
                "score": 0.0,
                "matched_target": "",
                "target_priority": 0,
                "target_kind": "none",
                "exact": False,
            }

        targets = cls._extract_ocr_targets(query)
        if targets:
            highest_priority = targets[0]["priority"]
            primary_targets = [
                target
                for target in targets
                if target["priority"] == highest_priority
            ]

            # Exact match of the highest-specificity target.
            for target in primary_targets:
                if target["text"] in region_norm:
                    return {
                        "score": 1.0,
                        "matched_target": target["text"],
                        "target_priority": target["priority"],
                        "target_kind": target["kind"],
                        "exact": True,
                    }

            # Partial matching is allowed for diagnostics/retrieval, but is
            # capped below the authoritative OCR-match threshold.
            best_partial = 0.0
            best_target = primary_targets[0]
            region_tokens = set(region_norm.split())
            for target in primary_targets:
                target_tokens = set(target["text"].split())
                coverage = len(
                    target_tokens & region_tokens
                ) / max(len(target_tokens), 1)
                sequence = SequenceMatcher(
                    None,
                    target["text"],
                    region_norm,
                ).ratio()
                partial = max(
                    0.70 * coverage + 0.30 * sequence,
                    sequence,
                )
                if partial > best_partial:
                    best_partial = partial
                    best_target = target

            return {
                "score": min(best_partial, 0.49),
                "matched_target": best_target["text"],
                "target_priority": best_target["priority"],
                "target_kind": best_target["kind"],
                "exact": False,
            }

        # Generic fallback only when the query has no structured OCR target.
        stop_words = {
            "where", "when", "what", "which", "is", "are", "was", "were",
            "the", "a", "an", "in", "on", "at", "to", "of", "for",
            "written", "write", "appear", "appears", "display", "displays",
            "show", "shows", "find", "line", "file",
            "أين", "متى", "ما", "في", "على", "الذي", "التي", "مكتوب", "يظهر",
        }
        query_tokens = {
            token
            for token in query_norm.split()
            if token not in stop_words and len(token) > 1
        }
        region_tokens = set(region_norm.split())
        if not query_tokens or not region_tokens:
            score = SequenceMatcher(
                None,
                query_norm,
                region_norm,
            ).ratio()
        else:
            intersection = query_tokens & region_tokens
            query_coverage = len(
                intersection
            ) / max(len(query_tokens), 1)
            region_precision = len(
                intersection
            ) / max(len(region_tokens), 1)
            score = max(
                SequenceMatcher(
                    None,
                    query_norm,
                    region_norm,
                ).ratio(),
                0.65 * region_precision
                + 0.35 * query_coverage,
            )

        return {
            "score": score,
            "matched_target": "",
            "target_priority": 0,
            "target_kind": "generic",
            "exact": False,
        }

    @classmethod
    def _token_similarity(cls, query: str, text: str) -> float:
        return float(
            cls._match_ocr_region(query, text)["score"]
        )

    def _ocr_grounding(self, plan, candidate):
        query = plan.query_for("ocr")
        observations = candidate.get("payload", {}).get(
            "ocr_observations", []
        )
        best = None

        for observation in observations:
            for region in observation.get("regions", []):
                region_text = region.get("text", "")
                match_info = self._match_ocr_region(
                    query,
                    region_text,
                )
                semantic_score = float(
                    match_info["score"]
                )
                detector_score = max(
                    0.0,
                    min(
                        float(region.get("confidence", 0.0)),
                        1.0,
                    ),
                )

                combined_score = min(
                    1.0,
                    0.90 * semantic_score
                    + 0.10 * detector_score,
                )
                bbox_norm = region.get("bbox_norm", []) or []
                bbox = region.get("bbox", []) or []

                # A structured target is authoritative only when the
                # highest-specificity phrase itself matched exactly.
                exact_target_match = bool(
                    match_info["exact"]
                    and match_info["target_priority"] >= 50
                )
                generic_match = bool(
                    match_info["target_kind"] == "generic"
                    and semantic_score >= 0.70
                )
                match = bool(
                    (exact_target_match or generic_match)
                    and (bbox_norm or bbox)
                )

                item = {
                    "match": match,
                    "confidence": combined_score,
                    "semantic_score": semantic_score,
                    "ocr_detector_confidence": detector_score,
                    "best_timestamp": float(
                        observation.get("timestamp", 0.0)
                    ),
                    "start_ts": float(
                        observation.get("timestamp", 0.0)
                    ),
                    "end_ts": float(
                        observation.get("timestamp", 0.0)
                    ) + 0.75,
                    "bbox_2d": bbox,
                    "bbox_norm": bbox_norm,
                    "frame_path": observation.get("path", ""),
                    "evidence_type": "ocr_region",
                    "reason": f"OCR region: {region_text}",
                    "matched_text": region_text,
                    "matched_target": match_info[
                        "matched_target"
                    ],
                    "target_priority": match_info[
                        "target_priority"
                    ],
                    "target_kind": match_info[
                        "target_kind"
                    ],
                    "exact_target_match": exact_target_match,
                }

                if best is None:
                    best = item
                elif item["match"] and not best.get("match"):
                    best = item
                elif (
                    item["match"] == best.get("match")
                    and item["confidence"]
                    > best.get("confidence", 0.0)
                ):
                    best = item

        return best

    @staticmethod
    def _select_ordered_frames(candidate, max_frames: int = 4):
        payload = candidate.get("payload", {})
        evidence = candidate.get("frame_evidence") or payload.get("frame_evidence", [])
        unique = {}
        for item in evidence:
            path = item.get("path", "")
            if path and os.path.exists(path):
                unique[path] = {
                    "path": path,
                    "timestamp": float(item.get("timestamp", 0.0)),
                    "width": int(item.get("width", 0)),
                    "height": int(item.get("height", 0)),
                }
        for frame in payload.get("keyframes", []):
            path = frame.get("path", "")
            if path and os.path.exists(path) and path not in unique:
                unique[path] = {
                    "path": path,
                    "timestamp": float(frame.get("timestamp", 0.0)),
                    "width": int(frame.get("width", 0)),
                    "height": int(frame.get("height", 0)),
                }
        frames = sorted(unique.values(), key=lambda item: item["timestamp"])
        if len(frames) > max_frames:
            indices = [round(i * (len(frames) - 1) / (max_frames - 1)) for i in range(max_frames)]
            frames = [frames[i] for i in indices]
        return frames

    @staticmethod
    def _parse_json(text: str):
        start, end = text.find("{"), text.rfind("}")
        if start < 0 or end <= start:
            return {}
        try:
            return json.loads(text[start:end + 1])
        except Exception:
            try:
                return json.loads(text[start:end + 1].replace("'", '"'))
            except Exception:
                return {}

    def _vlm_grounding(self, query, candidate):
        self.load_model()
        frames = self._select_ordered_frames(candidate)
        if not frames or self.model is None or self.processor is None:
            return None

        images = [Image.open(frame["path"]).convert("RGB") for frame in frames]
        timeline = [{"index": idx, "timestamp": frame["timestamp"]} for idx, frame in enumerate(frames)]
        content = []
        for idx, image in enumerate(images):
            content.append({"type": "text", "text": f"FRAME {idx}, timestamp={frames[idx]['timestamp']:.3f}s"})
            content.append({"type": "image", "image": image})
        content.append({
            "type": "text",
            "text": (
                f"Query: {query}\n"
                f"Transcript: {candidate.get('payload', {}).get('transcript', '')}\n"
                f"OCR: {candidate.get('payload', {}).get('ocr_text', '')}\n"
                f"Timeline: {json.dumps(timeline)}\n"
                "Return ONLY JSON with keys: match(boolean), event_start_index(integer), "
                "event_end_index(integer), best_frame_index(integer), bbox_2d([x1,y1,x2,y2] "
                "in normalized 0-1000 coordinates, or []), confidence(0-1), reason(string). "
                "Do not invent an event outside the ordered frames."
            ),
        })
        messages = [{"role": "user", "content": content}]
        prompt = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.processor(
            text=[prompt],
            images=images,
            return_tensors="pt",
            padding=True,
        )
        moved = {}
        for key, value in inputs.items():
            if torch is not None and isinstance(value, torch.Tensor):
                if torch.is_floating_point(value):
                    moved[key] = value.to(self.device, dtype=torch.float16)
                else:
                    moved[key] = value.to(self.device)
            else:
                moved[key] = value
        inputs = moved
        with torch.inference_mode():
            generated = self.model.generate(**inputs, max_new_tokens=160, do_sample=False)
        input_length = inputs["input_ids"].shape[1]
        decoded = self.processor.batch_decode(generated[:, input_length:], skip_special_tokens=True)[0].strip()
        data = self._parse_json(decoded)
        if not data:
            return None

        def safe_index(value, default=0):
            try:
                return min(max(int(value), 0), len(frames) - 1)
            except Exception:
                return default

        start_index = safe_index(data.get("event_start_index", 0))
        end_index = safe_index(data.get("event_end_index", start_index), start_index)
        if end_index < start_index:
            start_index, end_index = end_index, start_index
        best_index = safe_index(data.get("best_frame_index", start_index), start_index)
        bbox_norm = data.get("bbox_2d", []) or []
        width, height = frames[best_index]["width"], frames[best_index]["height"]
        bbox = []
        if len(bbox_norm) == 4 and width > 0 and height > 0:
            bbox = [
                int(round(bbox_norm[0] * width / 1000)),
                int(round(bbox_norm[1] * height / 1000)),
                int(round(bbox_norm[2] * width / 1000)),
                int(round(bbox_norm[3] * height / 1000)),
            ]
        return {
            "match": bool(data.get("match", False)),
            "confidence": max(0.0, min(float(data.get("confidence", 0.0)), 1.0)),
            "start_ts": frames[start_index]["timestamp"],
            "end_ts": frames[end_index]["timestamp"] + 0.75,
            "best_timestamp": frames[best_index]["timestamp"],
            "bbox_2d": bbox,
            "bbox_norm": bbox_norm if len(bbox_norm) == 4 else [],
            "frame_path": frames[best_index]["path"],
            "evidence_type": "qwen_vl_grounding",
            "reason": str(data.get("reason", "")),
            "raw_output": decoded,
        }

    def ground_candidates(
        self,
        query,
        plan,
        candidates,
        top_k: int = 5,
    ):
        if not candidates:
            return candidates

        if torch is not None and torch.cuda.is_available():
            gc.collect()
            torch.cuda.empty_cache()

        precomputed_ocr = {}
        for candidate in candidates:
            if "ocr" in plan.modalities:
                precomputed_ocr[candidate["candidate_id"]] = (
                    self._ocr_grounding(plan, candidate)
                )

        exact_ocr_available = any(
            evidence
            and evidence.get("match")
            and evidence.get("exact_target_match")
            and evidence.get("bbox_norm")
            for evidence in precomputed_ocr.values()
        )
        ocr_dominant_query = bool(
            plan.primary_modality == "ocr"
            or plan.modality_weights.get("ocr", 0.0)
            >= max(plan.modality_weights.values() or [0.0])
        )
        skip_vlm_globally = bool(
            exact_ocr_available
            and ocr_dominant_query
        )

        selected_log_count = 0

        for candidate_index, candidate in enumerate(candidates):
            ocr_grounding = precomputed_ocr.get(
                candidate["candidate_id"]
            )

            ocr_is_authoritative = bool(
                ocr_grounding
                and ocr_grounding.get("match")
                and ocr_grounding.get("exact_target_match")
                and ocr_grounding.get("bbox_norm")
            )

            vlm_grounding = None
            needs_vlm = bool(
                candidate_index < top_k
                and plan.requires_spatial_grounding
                and "visual" in plan.modalities
                and not ocr_is_authoritative
                and not skip_vlm_globally
            )

            if (
                ocr_is_authoritative
                and selected_log_count < 5
            ):
                print(
                    "[VerificationLayer] Exact OCR target selected "
                    f"target={ocr_grounding.get('matched_target')!r}, "
                    f"region={ocr_grounding.get('matched_text')!r}"
                )
                selected_log_count += 1

            if needs_vlm:
                try:
                    vlm_grounding = self._vlm_grounding(
                        plan.query_for("visual") or query,
                        candidate,
                    )
                except Exception as exc:
                    print(
                        f"[VerificationLayer] Grounding failed: {exc}"
                    )
                    if (
                        torch is not None
                        and torch.cuda.is_available()
                    ):
                        torch.cuda.empty_cache()

            if ocr_is_authoritative:
                grounding = ocr_grounding
            elif (
                ocr_grounding
                and ocr_grounding.get("match")
                and ocr_grounding.get("bbox_norm")
                and (
                    vlm_grounding is None
                    or not vlm_grounding.get("bbox_norm")
                )
            ):
                grounding = ocr_grounding
            elif vlm_grounding is not None:
                grounding = vlm_grounding
            elif ocr_grounding is not None:
                grounding = ocr_grounding
            else:
                grounding = None

            if grounding is None:
                evidence = candidate.get(
                    "frame_evidence",
                    [],
                )
                if evidence:
                    best = max(
                        evidence,
                        key=lambda item: item.get(
                            "score",
                            0.0,
                        ),
                    )
                    grounding = {
                        "match": True,
                        "confidence": max(
                            0.0,
                            min(
                                best.get("score", 0.0),
                                1.0,
                            ),
                        ),
                        "start_ts": best["timestamp"],
                        "end_ts": best["timestamp"] + 0.75,
                        "best_timestamp": best["timestamp"],
                        "bbox_2d": [],
                        "bbox_norm": [],
                        "frame_path": best.get("path", ""),
                        "evidence_type": "frame_vector",
                        "reason": "Fine frame-vector evidence",
                    }
                else:
                    grounding = {
                        "match": False,
                        "confidence": 0.0,
                        "bbox_2d": [],
                        "bbox_norm": [],
                        "evidence_type": "none",
                        "reason": "No spatial evidence",
                    }

            candidate["grounding"] = grounding
            candidate["ocr_grounding_score"] = (
                ocr_grounding.get("confidence", 0.0)
                if ocr_grounding
                else 0.0
            )
            candidate["visual_verification_score"] = (
                grounding.get("confidence", 0.0)
            )
            candidate["verified"] = grounding.get(
                "match",
                False,
            )

        return candidates

    def verify_candidates(self, query, candidates, top_k: int = 3):
        class LegacyPlan:
            modalities = ["visual"]
            requires_spatial_grounding = True
            def query_for(self, modality):
                return query
        return self.ground_candidates(query, LegacyPlan(), candidates, top_k)


Writing search/verification.py


In [17]:
%%writefile search/refiner.py
"""Clause-aware temporal algebra for speech, OCR, and visual intervals."""
import re
from difflib import SequenceMatcher
from typing import List, Dict, Any, Tuple, Optional

Span = Optional[Tuple[float, float]]


class TimestampRefiner:
    EXPANSION_MARGIN = 0.35

    @staticmethod
    def format_timestamp(seconds: float) -> str:
        seconds = max(0.0, float(seconds))
        hours = int(seconds // 3600)
        minutes = int((seconds % 3600) // 60)
        secs = int(seconds % 60)
        millis = int(round((seconds - int(seconds)) * 1000))
        if millis == 1000:
            secs += 1
            millis = 0
        return f"{hours:02d}:{minutes:02d}:{secs:02d}.{millis:03d}"

    @staticmethod
    def _tokens(text: str):
        stop = {
            "where", "when", "what", "the", "a", "an", "is", "are", "in", "on", "at",
            "show", "find", "moment", "متى", "أين", "في", "على", "التي", "الذي", "لحظة",
        }
        return [
            token for token in re.findall(r"[\w.-]+", text.casefold())
            if len(token) > 1 and token not in stop
        ]

    @staticmethod
    def _similarity(a: str, b: str) -> float:
        if a == b:
            return 1.0
        return SequenceMatcher(None, a, b).ratio()

    def find_speech_span(self, words: List[Dict[str, Any]], clause: str) -> Span:
        query_tokens = self._tokens(clause)
        if not words or not query_tokens:
            return None

        matched = []
        for index, word in enumerate(words):
            token = str(word.get("word", "")).strip(".,;:!?()[]{}\"'").casefold()
            similarity = max((self._similarity(token, query) for query in query_tokens), default=0.0)
            if similarity >= 0.72:
                matched.append((index, similarity))
        if not matched:
            return None

        best_window = None
        for start_pos in range(len(matched)):
            seen = set()
            for end_pos in range(start_pos, min(len(matched), start_pos + len(query_tokens) + 5)):
                word_index, _ = matched[end_pos]
                token = str(words[word_index].get("word", "")).casefold()
                for query_index, query in enumerate(query_tokens):
                    if self._similarity(token, query) >= 0.72:
                        seen.add(query_index)
                coverage = len(seen) / max(len(query_tokens), 1)
                if coverage >= min(0.5, 2 / max(len(query_tokens), 1)):
                    start_index = matched[start_pos][0]
                    end_index = word_index
                    span = (float(words[start_index]["start"]), float(words[end_index]["end"]))
                    score = coverage - 0.01 * (span[1] - span[0])
                    if best_window is None or score > best_window[0]:
                        best_window = (score, span)
        if best_window:
            return best_window[1]

        indices = [item[0] for item in matched]
        return float(words[min(indices)]["start"]), float(words[max(indices)]["end"])

    def find_ocr_span(self, observations, clause: str) -> Span:
        query_tokens = set(self._tokens(clause))
        if not observations or not query_tokens:
            return None
        matched_timestamps = []
        for observation in observations:
            text_tokens = set(self._tokens(observation.get("text", "")))
            if not text_tokens:
                continue
            coverage = len(query_tokens & text_tokens) / max(len(query_tokens), 1)
            sequence = SequenceMatcher(
                None, " ".join(sorted(query_tokens)), " ".join(sorted(text_tokens))
            ).ratio()
            if max(coverage, sequence) >= 0.35:
                matched_timestamps.append(float(observation.get("timestamp", 0.0)))
        if not matched_timestamps:
            return None
        return min(matched_timestamps), max(matched_timestamps) + 0.75

    @staticmethod
    def visual_span(candidate) -> Span:
        grounding = candidate.get("grounding", {}) or {}
        if grounding.get("match") and "start_ts" in grounding and "end_ts" in grounding:
            return float(grounding["start_ts"]), float(grounding["end_ts"])
        evidence = candidate.get("frame_evidence", [])
        timestamps = [
            float(item["timestamp"])
            for item in evidence
            if item.get("modality") == "visual" and item.get("score", 0.0) >= 0.2
        ]
        return (min(timestamps), max(timestamps) + 0.75) if timestamps else None

    @staticmethod
    def intersect(*spans: Span) -> Span:
        present = [span for span in spans if span is not None]
        if not present:
            return None
        start = max(span[0] for span in present)
        end = min(span[1] for span in present)
        return (start, end) if start < end else None

    @staticmethod
    def union(*spans: Span) -> Span:
        present = [span for span in spans if span is not None]
        if not present:
            return None
        return min(span[0] for span in present), max(span[1] for span in present)

    @staticmethod
    def iou(a: Span, b: Span) -> float:
        if a is None or b is None:
            return 0.0
        intersection = max(0.0, min(a[1], b[1]) - max(a[0], b[0]))
        union = max(a[1], b[1]) - min(a[0], b[0])
        return intersection / union if union > 0 else 0.0

    def refine_timestamps(self, candidate: Dict[str, Any], plan) -> Dict[str, Any]:
        payload = candidate.get("payload", {}) or {}
        clip_span = (
            float(payload.get("start_ts", 0.0)),
            float(payload.get("end_ts", 0.0)),
        )
        spans = {
            "speech": self.find_speech_span(
                payload.get("word_timestamps", []),
                plan.query_for("speech"),
            ) if "speech" in plan.modalities else None,
            "ocr": self.find_ocr_span(
                payload.get("ocr_observations", []),
                plan.query_for("ocr"),
            ) if "ocr" in plan.modalities else None,
            "visual": self.visual_span(candidate)
            if "visual" in plan.modalities
            else None,
        }

        grounding = candidate.get("grounding", {}) or {}
        exact_ocr_grounding = bool(
            grounding.get("match")
            and grounding.get("evidence_type") == "ocr_region"
            and grounding.get("exact_target_match")
            and "start_ts" in grounding
            and "end_ts" in grounding
        )
        if exact_ocr_grounding:
            spans["ocr"] = (
                float(grounding["start_ts"]),
                float(grounding["end_ts"]),
            )

        relation = plan.temporal_relation
        left = spans.get(plan.relation_left_modality)
        right = spans.get(plan.relation_right_modality)
        relation_satisfied = True
        relation_score = 1.0

        if relation == "while":
            refined = self.intersect(left, right)
            relation_satisfied = refined is not None and left is not None and right is not None
            relation_score = self.iou(left, right) if relation_satisfied else 0.0
        elif relation == "before":
            relation_satisfied = bool(left and right and left[1] <= right[0] + 0.25)
            refined = left
            gap = max(0.0, right[0] - left[1]) if left and right else 99.0
            relation_score = 1.0 / (1.0 + gap) if relation_satisfied else 0.0
        elif relation == "after":
            relation_satisfied = bool(left and right and left[0] >= right[1] - 0.25)
            refined = left
            gap = max(0.0, left[0] - right[1]) if left and right else 99.0
            relation_score = 1.0 / (1.0 + gap) if relation_satisfied else 0.0
        else:
            primary = spans.get(plan.primary_modality)
            refined = primary or self.intersect(*[spans[m] for m in plan.modalities])
            if refined is None:
                refined = self.union(*[spans[m] for m in plan.modalities])
            available = [span for span in spans.values() if span is not None]
            relation_score = 0.85 if available else 0.35

        if refined is None:
            refined = clip_span
            relation_satisfied = relation == "none"
            relation_score = 0.25 if relation == "none" else 0.0

        start = max(clip_span[0], refined[0] - self.EXPANSION_MARGIN)
        end = min(clip_span[1], refined[1] + self.EXPANSION_MARGIN)
        if end <= start:
            end = min(clip_span[1], start + 0.5)

        return {
            "refined_start": round(start, 3),
            "refined_end": round(end, 3),
            "formatted_start": self.format_timestamp(start),
            "formatted_end": self.format_timestamp(end),
            "temporal_score": max(0.0, min(relation_score, 1.0)),
            "relation_satisfied": relation_satisfied,
            "temporal_relation": relation,
            "modality_spans": {
                key: list(value) if value is not None else None for key, value in spans.items()
            },
            "spatial_evidence": candidate.get("grounding", {}),
        }


Writing search/refiner.py


In [18]:
%%writefile search/confidence.py
"""Five-signal composite confidence."""
class ConfidenceCalculator:
    def __init__(
        self,
        w_retrieval=0.25,
        w_cross=0.25,
        w_verification=0.20,
        w_temporal=0.20,
        w_spatial=0.10,
    ):
        weights = [w_retrieval, w_cross, w_verification, w_temporal, w_spatial]
        total = max(sum(weights), 1e-8)
        (
            self.w_retrieval,
            self.w_cross,
            self.w_verification,
            self.w_temporal,
            self.w_spatial,
        ) = [value / total for value in weights]

    @staticmethod
    def _clip(value):
        return max(0.0, min(float(value), 1.0))

    def calculate(
        self,
        retrieval_similarity,
        cross_encoder_score,
        verification_score,
        temporal_score,
        spatial_score,
        relation_required=False,
        relation_satisfied=True,
    ):
        values = {
            "retrieval_similarity": self._clip(retrieval_similarity),
            "cross_encoder_score": self._clip(cross_encoder_score),
            "verification_score": self._clip(verification_score),
            "temporal_score": self._clip(temporal_score),
            "spatial_score": self._clip(spatial_score),
        }
        final = (
            self.w_retrieval * values["retrieval_similarity"]
            + self.w_cross * values["cross_encoder_score"]
            + self.w_verification * values["verification_score"]
            + self.w_temporal * values["temporal_score"]
            + self.w_spatial * values["spatial_score"]
        )
        if relation_required and not relation_satisfied:
            final = min(final, 0.30)
        return self._clip(final), values


Writing search/confidence.py


In [19]:
%%writefile storage/metadata_store.py
"""SQLite exact metadata store for ASR words and spatial OCR regions."""
import os
import sqlite3
import json


class TemporalMetadataStore:
    def __init__(self, db_path="storage/temporal_metadata.db"):
        os.makedirs(os.path.dirname(db_path), exist_ok=True)
        self.conn = sqlite3.connect(db_path, check_same_thread=False)
        self._create_tables()

    def _create_tables(self):
        cursor = self.conn.cursor()
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS word_timestamps (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                clip_id TEXT NOT NULL,
                video_id TEXT NOT NULL,
                word TEXT NOT NULL,
                start_ts REAL NOT NULL,
                end_ts REAL NOT NULL
            )
        """)
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS ocr_regions (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                clip_id TEXT NOT NULL,
                video_id TEXT NOT NULL,
                frame_id TEXT,
                timestamp REAL NOT NULL,
                text TEXT NOT NULL,
                confidence REAL,
                bbox_json TEXT,
                bbox_norm_json TEXT,
                frame_path TEXT
            )
        """)
        cursor.execute("CREATE INDEX IF NOT EXISTS idx_word_video_clip ON word_timestamps(video_id, clip_id)")
        cursor.execute("CREATE INDEX IF NOT EXISTS idx_ocr_video_clip ON ocr_regions(video_id, clip_id)")
        cursor.execute("CREATE INDEX IF NOT EXISTS idx_ocr_timestamp ON ocr_regions(timestamp)")
        self.conn.commit()

    def store_word_timestamps(self, clip_id, video_id, words):
        rows = [
            (clip_id, video_id, str(word["word"]), float(word["start"]), float(word["end"]))
            for word in words if {"word", "start", "end"} <= set(word)
        ]
        if rows:
            self.conn.executemany(
                "INSERT INTO word_timestamps (clip_id, video_id, word, start_ts, end_ts) VALUES (?, ?, ?, ?, ?)",
                rows,
            )
            self.conn.commit()

    def store_ocr_observations(self, clip_id, video_id, observations):
        rows = []
        for observation in observations:
            for region in observation.get("regions", []):
                rows.append((
                    clip_id,
                    video_id,
                    observation.get("frame_id", ""),
                    float(observation.get("timestamp", 0.0)),
                    region.get("text", ""),
                    float(region.get("confidence", 0.0)),
                    json.dumps(region.get("bbox", [])),
                    json.dumps(region.get("bbox_norm", [])),
                    observation.get("path", ""),
                ))
        if rows:
            self.conn.executemany(
                """INSERT INTO ocr_regions
                (clip_id, video_id, frame_id, timestamp, text, confidence, bbox_json, bbox_norm_json, frame_path)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)""",
                rows,
            )
            self.conn.commit()

    def clear_video(self, video_id):
        self.conn.execute("DELETE FROM word_timestamps WHERE video_id=?", (video_id,))
        self.conn.execute("DELETE FROM ocr_regions WHERE video_id=?", (video_id,))
        self.conn.commit()


Writing storage/metadata_store.py


In [20]:
%%writefile api/schemas.py
from pydantic import BaseModel, Field
from typing import Optional, Dict, Any


class SearchQueryRequest(BaseModel):
    query: str = Field(..., description="الاستعلام النصي المركب")
    video_id: Optional[str] = Field(None, description="معرف الفيديو")


class IngestRequest(BaseModel):
    video_url: Optional[str] = None
    video_path: Optional[str] = None
    video_id: str = "default_video"


class StrictSearchResponse(BaseModel):
    start_timestamp: str
    end_timestamp: str
    confidence_score: float


class DetailedSearchResponse(StrictSearchResponse):
    search_id: str = ""
    candidate_id: str = ""
    video_id: str = ""
    clip_id: str = ""
    speech_excerpt: str = ""
    ocr_text: str = ""
    retrieval_method: str = ""
    temporal_relation: str = "none"
    relation_satisfied: bool = True
    modality_spans: Dict[str, Any] = Field(default_factory=dict)
    spatial_evidence: Dict[str, Any] = Field(default_factory=dict)
    scores: Dict[str, float] = Field(default_factory=dict)
    stage_latencies: Dict[str, float] = Field(default_factory=dict)


class IngestResponse(BaseModel):
    status: str
    video_id: str
    total_clips: int
    total_frames: int = 0
    segmentation_method: str = "scene_plus_asr"
    metrics: Dict[str, Any] = Field(default_factory=dict)


Writing api/schemas.py


In [21]:
%%writefile monitoring/performance.py
"""Persistent ingestion/search latency and RAM/VRAM instrumentation."""
import json
import os
import time
from contextlib import contextmanager

try:
    import psutil
except Exception:
    psutil = None

try:
    import torch
except Exception:
    torch = None


class PerformanceRegistry:
    def __init__(self, path="storage/performance_metrics.json"):
        self.path = path
        os.makedirs(os.path.dirname(path), exist_ok=True)
        self.data = self._load()

    def _load(self):
        if os.path.exists(self.path):
            try:
                with open(self.path, "r", encoding="utf-8") as handle:
                    return json.load(handle)
            except Exception:
                pass
        return {"ingestions": [], "searches": []}

    def _save(self):
        with open(self.path, "w", encoding="utf-8") as handle:
            json.dump(self.data, handle, ensure_ascii=False, indent=2)

    @staticmethod
    def memory_snapshot():
        rss_mb = 0.0
        if psutil is not None:
            rss_mb = psutil.Process(os.getpid()).memory_info().rss / (1024 ** 2)
        gpu_memory = {}
        total_allocated = 0.0
        if torch is not None and torch.cuda.is_available():
            for index in range(torch.cuda.device_count()):
                allocated = torch.cuda.memory_allocated(index) / (1024 ** 2)
                reserved = torch.cuda.memory_reserved(index) / (1024 ** 2)
                gpu_memory[f"gpu_{index}"] = {
                    "allocated_mb": round(allocated, 2),
                    "reserved_mb": round(reserved, 2),
                }
                total_allocated += allocated
        return {
            "ram_mb": round(rss_mb, 2),
            "vram_mb": round(total_allocated, 2),
            "gpu_memory": gpu_memory,
        }

    @contextmanager
    def stage(self, stage_metrics, name):
        before = self.memory_snapshot()
        if torch is not None and torch.cuda.is_available():
            for index in range(torch.cuda.device_count()):
                torch.cuda.reset_peak_memory_stats(index)
        started = time.perf_counter()
        try:
            yield
        finally:
            duration = time.perf_counter() - started
            after = self.memory_snapshot()
            peak_by_gpu = {}
            peak_vram = 0.0
            if torch is not None and torch.cuda.is_available():
                for index in range(torch.cuda.device_count()):
                    peak = torch.cuda.max_memory_allocated(index) / (1024 ** 2)
                    peak_by_gpu[f"gpu_{index}"] = round(peak, 2)
                    peak_vram += peak
            stage_metrics[name] = {
                "latency_sec": round(duration, 4),
                "ram_before_mb": before["ram_mb"],
                "ram_after_mb": after["ram_mb"],
                "max_observed_ram_mb": max(
                    before["ram_mb"],
                    after["ram_mb"],
                ),
                "vram_after_mb": after["vram_mb"],
                "peak_vram_mb": round(peak_vram, 2),
                "peak_vram_by_gpu_mb": peak_by_gpu,
            }

    def record_ingestion(self, record):
        self.data["ingestions"].append(record)
        self.data["ingestions"] = self.data["ingestions"][-50:]
        self._save()

    def record_search(self, record):
        self.data["searches"].append(record)
        self.data["searches"] = self.data["searches"][-500:]
        self._save()

    @staticmethod
    def _percentile(values, percentile):
        if not values:
            return 0.0
        ordered = sorted(values)
        index = min(len(ordered) - 1, max(0, round((percentile / 100) * (len(ordered) - 1))))
        return ordered[index]

    def summary(self):
        latencies = [
            float(item.get("latency_sec", 0.0))
            for item in self.data["searches"]
        ]
        enough_samples = len(latencies) >= 5

        return {
            "ingestion_runs": len(self.data["ingestions"]),
            "search_runs": len(self.data["searches"]),
            "search_latency_sample_size": len(latencies),
            "search_latency_p50_sec": (
                round(self._percentile(latencies, 50), 4)
                if enough_samples
                else None
            ),
            "search_latency_p95_sec": (
                round(self._percentile(latencies, 95), 4)
                if enough_samples
                else None
            ),
            "search_latency_max_sec": (
                round(max(latencies), 4)
                if latencies
                else None
            ),
            "latest_ingestion": (
                self.data["ingestions"][-1]
                if self.data["ingestions"]
                else {}
            ),
            "latest_search": (
                self.data["searches"][-1]
                if self.data["searches"]
                else {}
            ),
        }

    def arabic_markdown(self):
        summary = self.summary()
        latest_ingestion = summary["latest_ingestion"]
        return f"""# تقرير أداء Spatio-Temporal Video RAG

| المقياس | القيمة |
|---|---:|
| عدد مرات الفهرسة المقاسة | {summary['ingestion_runs']} |
| عدد استعلامات البحث المقاسة | {summary['search_runs']} |
| Search latency P50 | {summary['search_latency_p50_sec']} ثانية |
| Search latency P95 | {summary['search_latency_p95_sec']} ثانية |
| أعلى Search latency | {summary['search_latency_max_sec']} ثانية |
| آخر عدد Clips | {latest_ingestion.get('total_clips', 0)} |
| آخر عدد Frames | {latest_ingestion.get('total_frames', 0)} |
| مدة آخر فيديو | {latest_ingestion.get('video_duration_sec', 0)} ثانية |
| زمن آخر فهرسة | {latest_ingestion.get('latency_sec', 0)} ثانية |
| Peak VRAM للفهرسة | {latest_ingestion.get('peak_vram_mb', 0)} MB |
| أعلى RAM مرصود للفهرسة | {latest_ingestion.get('max_observed_ram_mb', 0)} MB |

## آخر فهرسة

```json
{json.dumps(latest_ingestion, ensure_ascii=False, indent=2)}
```

## آخر بحث

```json
{json.dumps(summary['latest_search'], ensure_ascii=False, indent=2)}
```
"""


Writing monitoring/performance.py


In [22]:
%%writefile evaluation/metrics.py
"""Temporal grounding evaluation metrics."""
def temporal_iou(predicted, truth):
    intersection = max(0.0, min(predicted[1], truth[1]) - max(predicted[0], truth[0]))
    union = max(predicted[1], truth[1]) - min(predicted[0], truth[0])
    return intersection / union if union > 0 else 0.0


def evaluate_predictions(cases):
    rows = []
    for case in cases:
        score = temporal_iou(tuple(case["predicted"]), tuple(case["truth"]))
        rows.append({**case, "temporal_iou": score, "iou_at_0_5": score >= 0.5})
    mean_iou = sum(row["temporal_iou"] for row in rows) / max(len(rows), 1)
    recall = sum(row["iou_at_0_5"] for row in rows) / max(len(rows), 1)
    return {
        "cases": rows,
        "mean_temporal_iou": mean_iou,
        "recall_at_iou_0_5": recall,
    }


Writing evaluation/metrics.py


In [23]:
%%writefile runtime/model_manager.py
"""Central lazy/staged model lifecycle for single or dual T4 runtimes."""
import gc
import os
from typing import Optional

from search.device_detector import (
    configured_devices,
    device_diagnostics,
    release_cuda,
)


class RuntimeModelManager:
    def __init__(self):
        self.devices = configured_devices()
        self.transcriber = None
        self.ocr = None
        self.embedder = None
        self.query_planner = None
        self.reranker = None
        self.verifier = None

    def diagnostics(self):
        return device_diagnostics()

    def get_transcriber(self):
        if self.transcriber is None:
            from ingestion.transcriber import AudioTranscriber
            self.transcriber = AudioTranscriber(
                device=self.devices["ingest"],
                autoload=False,
            )
        return self.transcriber

    def get_ocr(self):
        if self.ocr is None:
            from ingestion.ocr_extractor import OCRExtractor
            languages = [
                item.strip()
                for item in os.environ.get("VIDEO_RAG_OCR_LANGS", "en").split(",")
                if item.strip()
            ]
            # EasyOCR internally uses DataParallel whose source device is
            # cuda:0. OCR is sequentially executed after Whisper is released,
            # therefore the primary ingestion GPU is safe and available.
            ocr_device = os.environ.get(
                "VIDEO_RAG_OCR_DEVICE",
                self.devices["ingest"],
            )
            if str(ocr_device).startswith("cuda") and ocr_device != "cuda:0":
                print(
                    f"[RuntimeModelManager] Remapping OCR {ocr_device} -> cuda:0 "
                    f"for EasyOCR DataParallel compatibility"
                )
                ocr_device = "cuda:0"
            self.ocr = OCRExtractor(
                languages=languages or ["en"],
                device=ocr_device,
                autoload=False,
                max_frames_per_clip=int(
                    os.environ.get("VIDEO_RAG_OCR_FRAMES_PER_CLIP", "6")
                ),
                canvas_size=int(
                    os.environ.get("VIDEO_RAG_OCR_CANVAS_SIZE", "1280")
                ),
            )
        return self.ocr

    def get_embedder(self):
        if self.embedder is None:
            from ingestion.embedder import MultimodalEmbedder
            self.embedder = MultimodalEmbedder(
                device=self.devices["ingest"],
                autoload=False,
            )
        return self.embedder

    def get_query_planner(self):
        if self.query_planner is None:
            from search.query_planner import QueryPlanner
            self.query_planner = QueryPlanner(
                device=self.devices["search"],
            )
        return self.query_planner

    def get_reranker(self):
        if self.reranker is None:
            from search.reranker import CrossEncoderReranker
            self.reranker = CrossEncoderReranker(
                device=self.devices["search"],
                autoload=False,
            )
        return self.reranker

    def get_verifier(self):
        if self.verifier is None:
            from search.verification import VideoVerificationLayer
            self.verifier = VideoVerificationLayer(
                enable=True,
                device=self.devices["search"],
                autoload=False,
            )
        return self.verifier

    def release(self, name):
        obj = getattr(self, name, None)
        if obj is not None:
            release = getattr(obj, "release_model", None)
            if release is None:
                release = getattr(obj, "release_models", None)
            if callable(release):
                release()
        setattr(self, name, None)
        gc.collect()
        release_cuda()

    def release_ingestion(self):
        for name in ("transcriber", "ocr", "embedder"):
            self.release(name)

    def release_search(self):
        for name in ("query_planner", "reranker", "verifier", "embedder"):
            self.release(name)

    def release_all(self):
        self.release_ingestion()
        self.release_search()


Writing runtime/model_manager.py


In [24]:
%%writefile api/main.py
"""Spatio-Temporal Multimodal Video RAG application."""
import os
import time
import uuid
from dataclasses import dataclass

from fastapi import FastAPI, HTTPException
from fastapi.responses import PlainTextResponse

from api.schemas import (
    SearchQueryRequest,
    StrictSearchResponse,
    DetailedSearchResponse,
    IngestResponse,
    IngestRequest,
)
from ingestion.video_processor import VideoProcessor
from ingestion.indexer import QdrantIndexer
from search.retriever import MultimodalRetriever
from search.refiner import TimestampRefiner
from search.confidence import ConfidenceCalculator
from search.temporal_reasoner import TemporalReasoner
from storage.metadata_store import TemporalMetadataStore
from monitoring.performance import PerformanceRegistry
from runtime.model_manager import RuntimeModelManager

app = FastAPI(
    title="Spatio-Temporal Video RAG",
    description=(
        "Visual, speech, OCR, spatial grounding, and temporal reasoning"
    ),
    version="1.0.0",
)

# Lightweight objects only. No neural model is loaded during import.
video_processor = VideoProcessor(frame_sample_fps=1.0)
indexer = QdrantIndexer(
    location=os.environ.get(
        "VIDEO_RAG_QDRANT_PATH",
        "storage/qdrant_db",
    )
)
metadata_store = TemporalMetadataStore()
refiner = TimestampRefiner()
confidence_calculator = ConfidenceCalculator()
temporal_reasoner = TemporalReasoner()
performance_registry = PerformanceRegistry()
model_manager = RuntimeModelManager()



def _derive_video_id(target):
    base = os.path.splitext(os.path.basename(target or ""))[0]
    clean = "".join(
        character if character.isalnum() else "_"
        for character in base
    ).strip("_")
    return clean or f"video_{uuid.uuid4().hex[:8]}"


@app.get("/runtime")
def runtime_status():
    return {
        "models_loaded": {
            name: getattr(model_manager, name) is not None
            for name in (
                "transcriber",
                "ocr",
                "embedder",
                "query_planner",
                "reranker",
                "verifier",
            )
        },
        "devices": model_manager.diagnostics(),
    }


@app.post("/runtime/release")
def release_runtime_models():
    model_manager.release_all()
    return runtime_status()


@app.post("/ingest", response_model=IngestResponse)
def ingest_video_endpoint(request: IngestRequest):
    target = request.video_path or request.video_url
    if not target:
        raise HTTPException(400, "video_path or video_url is required")
    if target.startswith(("http://", "https://")):
        raise HTTPException(400, "Upload the video locally in this notebook")
    if not os.path.exists(target):
        raise HTTPException(400, f"Video does not exist: {target}")

    video_id = (
        request.video_id
        if request.video_id != "default_video"
        else _derive_video_id(target)
    )
    temp_dir = os.path.join("storage", "processed", video_id)
    keyframe_dir = os.path.join(temp_dir, "keyframes")
    audio_path = os.path.join(temp_dir, "audio.wav")
    os.makedirs(temp_dir, exist_ok=True)

    stage_metrics = {}
    process_started = time.perf_counter()
    metadata = video_processor.video_metadata(target)

    try:
        # CPU stage.
        with performance_registry.stage(stage_metrics, "audio_extraction"):
            video_processor.extract_audio(target, audio_path)

        # GPU 0 stage 1: Whisper only.
        transcriber = model_manager.get_transcriber()
        with performance_registry.stage(stage_metrics, "asr"):
            words = transcriber.transcribe_audio(audio_path)
            asr_boundaries = transcriber.detect_sentence_boundaries(words)
        transcriber.release_model()

        # CPU stage.
        with performance_registry.stage(
            stage_metrics,
            "segmentation_and_frames",
        ):
            clips = video_processor.process_video(
                target,
                asr_boundaries,
                keyframe_dir,
            )
            clips = transcriber.align_transcripts_to_chunks(clips, words)
        model_manager.release("transcriber")

        # OCR stage after the transcription model has been released.
        ocr_engine = model_manager.get_ocr()
        print(
            f"[Ingestion] Starting stable OCR on {ocr_engine.device}; "
            f"clips={len(clips)}"
        )
        ocr_engine.load_model()
        ocr_processed_frames = 0
        ocr_nonempty_frames = 0
        ocr_region_count = 0
        with performance_registry.stage(stage_metrics, "spatial_ocr"):
            for clip in clips:
                result = ocr_engine.process_clip_ocr_detailed(
                    clip.get("keyframes", [])
                )
                clip["ocr_text"] = result["combined_text"]
                clip["ocr_observations"] = result["observations"]
                clip["ocr_intervals"] = result["intervals"]
                ocr_processed_frames += result.get("processed_frames", 0)
                ocr_nonempty_frames += result.get("nonempty_frames", 0)
                ocr_region_count += result.get("region_count", 0)
        ocr_backend_name = ocr_engine.backend
        print(
            f"[Ingestion] OCR backend={ocr_backend_name} "
            f"processed={ocr_processed_frames} "
            f"nonempty={ocr_nonempty_frames} "
            f"regions={ocr_region_count}"
        )
        model_manager.release("ocr")

        # GPU 0 stage 2: BGE-M3 + SigLIP2 only.
        embedder = model_manager.get_embedder()
        with performance_registry.stage(
            stage_metrics,
            "multimodal_embeddings",
        ):
            clips = embedder.embed_clips(clips)

        # Index before releasing because vectors live in Python lists.
        with performance_registry.stage(stage_metrics, "qdrant_indexing"):
            metadata_store.clear_video(video_id)
            total_clips = indexer.index_clips(
                video_id,
                clips,
                source_video_path=target,
            )
            for clip in clips:
                metadata_store.store_word_timestamps(
                    clip["clip_id"],
                    video_id,
                    clip.get("word_timestamps", []),
                )
                metadata_store.store_ocr_observations(
                    clip["clip_id"],
                    video_id,
                    clip.get("ocr_observations", []),
                )
        model_manager.release("embedder")

    except Exception:
        model_manager.release_ingestion()
        raise

    elapsed = time.perf_counter() - process_started
    snapshots = [
        value for value in stage_metrics.values()
        if isinstance(value, dict)
    ]
    record = {
        "video_id": video_id,
        "video_duration_sec": round(metadata["duration"], 3),
        "total_clips": total_clips,
        "total_frames": indexer.indexed_frame_count(video_id),
        "latency_sec": round(elapsed, 3),
        "seconds_per_video_minute": round(
            elapsed / max(metadata["duration"] / 60.0, 1e-8),
            3,
        ),
        "max_observed_ram_mb": max(
            (
                item.get("max_observed_ram_mb", 0.0)
                for item in snapshots
            ),
            default=0.0,
        ),
        "peak_vram_mb": max(
            (item.get("peak_vram_mb", 0.0) for item in snapshots),
            default=0.0,
        ),
        "stages": stage_metrics,
        "device_assignment": model_manager.devices,
        "ocr_backend": ocr_backend_name,
        "ocr_processed_frames": ocr_processed_frames,
        "ocr_nonempty_frames": ocr_nonempty_frames,
        "ocr_region_count": ocr_region_count,
    }
    performance_registry.record_ingestion(record)
    return IngestResponse(
        status="success",
        video_id=video_id,
        total_clips=total_clips,
        total_frames=record["total_frames"],
        metrics=record,
    )


def ingest_video(video_path=None, video_url=None, video_id=None):
    return ingest_video_endpoint(
        IngestRequest(
            video_path=video_path,
            video_url=video_url,
            video_id=video_id
            or _derive_video_id(video_path or video_url or ""),
        )
    )


@dataclass(frozen=True)
class SearchExecution:
    search_id: str
    best: dict
    refined: dict
    plan: object
    stage_latencies: dict


def _assert_integrity(candidate):
    payload = candidate.get("payload", {}) or {}
    expected = (
        f"{payload.get('video_id', '')}:"
        f"{payload.get('clip_id', '')}"
    )
    if candidate.get("candidate_id") != expected:
        raise RuntimeError(
            f"Candidate integrity error: "
            f"{candidate.get('candidate_id')} != {expected}"
        )


def execute_search_once(request: SearchQueryRequest) -> SearchExecution:
    query = request.query.strip()
    video_id = request.video_id
    if not query:
        raise HTTPException(400, "Query cannot be empty")
    status = indexer.index_status(video_id)
    if not status["ready"]:
        raise HTTPException(409, f"Index is not ready: {status}")

    stages = {}
    search_started = time.perf_counter()

    try:
        # Query planning stage. The deterministic parser is the default.
        stage = time.perf_counter()
        planner = model_manager.get_query_planner()
        plan = planner.plan(query)
        stages["query_planning"] = time.perf_counter() - stage
        model_manager.release("query_planner")

        # GPU 0 stage: BGE-M3 + SigLIP2 query embeddings.
        stage = time.perf_counter()
        embedder = model_manager.get_embedder()
        retriever = MultimodalRetriever(indexer, embedder)
        candidates = retriever.search(
            plan,
            video_id=video_id,
            top_k=30,
        )
        candidates = retriever.attach_fine_frame_evidence(
            plan,
            candidates,
            video_id=video_id,
        )
        stages["coarse_and_fine_retrieval"] = (
            time.perf_counter() - stage
        )
        model_manager.release("embedder")
        if not candidates:
            raise HTTPException(404, "No candidates returned")

        # GPU 1 stage 2: BGE reranker only.
        stage = time.perf_counter()
        reranker = model_manager.get_reranker()
        candidates = reranker.rerank(query, candidates)
        stages["cross_encoder_rerank"] = (
            time.perf_counter() - stage
        )
        model_manager.release("reranker")

        # Spatial verification stage. Qwen-VL is loaded only when required.
        stage = time.perf_counter()
        verifier = model_manager.get_verifier()
        candidates = verifier.ground_candidates(
            query,
            plan,
            candidates,
            top_k=5,
        )
        stages["spatial_temporal_grounding"] = (
            time.perf_counter() - stage
        )
        model_manager.release("verifier")

        # CPU temporal algebra and confidence.
        stage = time.perf_counter()
        evaluated = []
        for candidate in candidates[:30]:
            _assert_integrity(candidate)
            refined = refiner.refine_timestamps(candidate, plan)
            grounding = candidate.get("grounding", {}) or {}
            spatial_score = (
                grounding.get("confidence", 0.0)
                if grounding.get("bbox_norm")
                else 0.6 * grounding.get("confidence", 0.0)
            )
            confidence, breakdown = confidence_calculator.calculate(
                candidate.get("retrieval_similarity", 0.0),
                candidate.get("cross_score", 0.0),
                candidate.get("visual_verification_score", 0.0),
                refined.get("temporal_score", 0.0),
                spatial_score,
                relation_required=plan.temporal_relation != "none",
                relation_satisfied=refined.get(
                    "relation_satisfied",
                    True,
                ),
            )
            candidate["refined"] = refined
            candidate["final_confidence"] = confidence
            candidate["score_breakdown"] = breakdown
            evaluated.append(candidate)

        ordered = temporal_reasoner.select(
            evaluated,
            plan.temporal_selector,
        )
        if not ordered:
            raise HTTPException(
                404,
                f"No candidate satisfies temporal relation {plan.temporal_relation!r}.",
            )
        best = ordered[0]
        refined = best["refined"]
        stages["temporal_algebra_and_selection"] = (
            time.perf_counter() - stage
        )

    except Exception:
        model_manager.release_search()
        raise

    elapsed = time.perf_counter() - search_started
    search_id = uuid.uuid4().hex
    performance_registry.record_search({
        "search_id": search_id,
        "video_id": video_id,
        "query": query,
        "candidate_id": best["candidate_id"],
        "latency_sec": round(elapsed, 4),
        "confidence": round(best["final_confidence"], 4),
        "relation": plan.temporal_relation,
        "relation_satisfied": refined.get(
            "relation_satisfied",
            True,
        ),
        "stage_latencies": {
            key: round(value, 4)
            for key, value in stages.items()
        },
        "device_assignment": model_manager.devices,
        **performance_registry.memory_snapshot(),
    })
    return SearchExecution(
        search_id,
        best,
        refined,
        plan,
        stages,
    )


def _strict(execution):
    return StrictSearchResponse(
        start_timestamp=execution.refined["formatted_start"],
        end_timestamp=execution.refined["formatted_end"],
        confidence_score=round(
            execution.best["final_confidence"],
            4,
        ),
    )


def _detailed(execution):
    candidate = execution.best
    refined = execution.refined
    plan = execution.plan
    payload = candidate.get("payload", {})
    return DetailedSearchResponse(
        start_timestamp=refined["formatted_start"],
        end_timestamp=refined["formatted_end"],
        confidence_score=round(
            candidate["final_confidence"],
            4,
        ),
        search_id=execution.search_id,
        candidate_id=candidate["candidate_id"],
        video_id=payload.get("video_id", ""),
        clip_id=payload.get("clip_id", ""),
        speech_excerpt=payload.get("transcript", ""),
        ocr_text=payload.get("ocr_text", ""),
        retrieval_method=(
            "coarse_clip_rrf+fine_frame+"
            + str(
                refined.get(
                    "spatial_evidence",
                    {},
                ).get(
                    "evidence_type",
                    "none",
                )
            )
        ),
        temporal_relation=plan.temporal_relation,
        relation_satisfied=refined.get(
            "relation_satisfied",
            True,
        ),
        modality_spans=refined.get("modality_spans", {}),
        spatial_evidence=refined.get(
            "spatial_evidence",
            {},
        ),
        scores=candidate.get("score_breakdown", {}),
        stage_latencies={
            key: round(value, 4)
            for key, value in execution.stage_latencies.items()
        },
    )


@app.post("/search", response_model=StrictSearchResponse)
def search_video_segment(request: SearchQueryRequest):
    return _strict(execute_search_once(request))


@app.post(
    "/search/detailed",
    response_model=DetailedSearchResponse,
)
def search_video_detailed(request: SearchQueryRequest):
    return _detailed(execute_search_once(request))


def search_video_both(request):
    execution = execute_search_once(request)
    return _strict(execution), _detailed(execution)


@app.get("/metrics")
def metrics():
    return performance_registry.summary()


@app.get("/metrics/report", response_class=PlainTextResponse)
def metrics_report():
    return performance_registry.arabic_markdown()


Writing api/main.py


In [25]:
# Runtime bootstrap
import importlib
import sys

for module_name in list(sys.modules):
    if module_name in {
        "api",
        "search",
        "ingestion",
        "storage",
        "runtime",
        "monitoring",
        "evaluation",
    } or module_name.startswith((
        "api.",
        "search.",
        "ingestion.",
        "storage.",
        "runtime.",
        "monitoring.",
        "evaluation.",
    )):
        sys.modules.pop(module_name, None)

importlib.invalidate_caches()

from api.main import (
    ingest_video,
    search_video_both,
    search_video_segment,
    search_video_detailed,
    indexer,
    model_manager,
    performance_registry,
    app,
)
from api.schemas import SearchQueryRequest

runtime_info = model_manager.diagnostics()
print("Runtime information:", runtime_info)

assert runtime_info["gpu_count"] >= 1, (
    "Select a GPU runtime before continuing."
)

if runtime_info["gpu_count"] >= 2:
    print(
        "Dual GPU runtime detected. "
        "Ingestion and embeddings use cuda:0. "
        "Search models use cuda:1."
    )
else:
    print(
        "Single GPU runtime detected. "
        "Staged model loading will be used."
    )

print("Index status:", indexer.index_status())
print("API imported without loading heavyweight neural models.")


Runtime information: {'cuda_available': True, 'gpu_count': 2, 'assignment': {'ingest': 'cuda:0', 'search': 'cuda:1', 'ocr': 'cuda:0'}, 'gpus': [{'index': 0, 'name': 'Tesla T4', 'capability': (7, 5), 'dtype': 'torch.float16', 'free_gb': 14.46, 'total_gb': 14.56}, {'index': 1, 'name': 'Tesla T4', 'capability': (7, 5), 'dtype': 'torch.float16', 'free_gb': 14.46, 'total_gb': 14.56}]}
Dual GPU runtime detected. Ingestion and embeddings use cuda:0. Search models use cuda:1.
Index status: {'clip_collection': 'spatio_temporal_clips', 'frame_collection': 'spatio_temporal_frames', 'video_id': '*', 'indexed_clips': 0, 'indexed_frames': 0, 'qdrant_connected': True, 'ready': False}
API imported without loading heavyweight neural models.


In [26]:
# Runtime device assignment validation
runtime = model_manager.diagnostics()
print(runtime)

assert runtime["assignment"]["ingest"] == "cuda:0"
assert runtime["assignment"]["ocr"] == "cuda:0"

if runtime["gpu_count"] >= 2:
    assert runtime["assignment"]["search"] == "cuda:1"
    print("Dual GPU assignment validated.")
else:
    assert runtime["assignment"]["search"] == "cuda:0"
    print("Single GPU staged assignment validated.")


{'cuda_available': True, 'gpu_count': 2, 'assignment': {'ingest': 'cuda:0', 'search': 'cuda:1', 'ocr': 'cuda:0'}, 'gpus': [{'index': 0, 'name': 'Tesla T4', 'capability': (7, 5), 'dtype': 'torch.float16', 'free_gb': 14.46, 'total_gb': 14.56}, {'index': 1, 'name': 'Tesla T4', 'capability': (7, 5), 'dtype': 'torch.float16', 'free_gb': 14.46, 'total_gb': 14.56}]}
Dual GPU assignment validated.


In [27]:
# Video ingestion and indexing
import os

VIDEO_ID = os.environ.get(
    "VIDEO_RAG_VIDEO_ID",
    "my_english_demo",
)
VIDEO_PATH = os.environ.get(
    "VIDEO_RAG_VIDEO_PATH",
    "/kaggle/input/datasets/basharalmuntaser/mydata/docker_in_100s.mp4",
)

if not os.path.exists(VIDEO_PATH):
    try:
        from google.colab import files

        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("No video was uploaded.")
        VIDEO_PATH = f"/content/{next(iter(uploaded))}"
    except ImportError as exc:
        raise FileNotFoundError(
            "Set VIDEO_RAG_VIDEO_PATH to an existing local video file."
        ) from exc

print("Runtime before ingestion:", model_manager.diagnostics())

ingest_result = ingest_video(
    video_path=VIDEO_PATH,
    video_id=VIDEO_ID,
)
ingestion_data = ingest_result.model_dump()
ingestion_metrics = ingestion_data["metrics"]

ingestion_summary = {
    "status": ingestion_data["status"],
    "video_id": ingestion_data["video_id"],
    "total_clips": ingestion_data["total_clips"],
    "total_frames": ingestion_data["total_frames"],
    "latency_sec": ingestion_metrics["latency_sec"],
    "ocr_backend": ingestion_metrics["ocr_backend"],
    "ocr_processed_frames": ingestion_metrics[
        "ocr_processed_frames"
    ],
    "ocr_nonempty_frames": ingestion_metrics[
        "ocr_nonempty_frames"
    ],
    "ocr_region_count": ingestion_metrics[
        "ocr_region_count"
    ],
    "peak_vram_mb": ingestion_metrics["peak_vram_mb"],
    "max_observed_ram_mb": ingestion_metrics[
        "max_observed_ram_mb"
    ],
}

print("Ingestion summary:", ingestion_summary)
print("Runtime after ingestion:", model_manager.diagnostics())
print("Index status:", indexer.index_status(VIDEO_ID))

assert ingestion_data["status"] == "success"
assert ingestion_metrics["ocr_backend"] == "easyocr"
assert ingestion_metrics["ocr_processed_frames"] > 0
assert ingestion_metrics["ocr_nonempty_frames"] > 0
assert indexer.index_status(VIDEO_ID)["ready"] is True


Runtime before ingestion: {'cuda_available': True, 'gpu_count': 2, 'assignment': {'ingest': 'cuda:0', 'search': 'cuda:1', 'ocr': 'cuda:0'}, 'gpus': [{'index': 0, 'name': 'Tesla T4', 'capability': (7, 5), 'dtype': 'torch.float16', 'free_gb': 14.46, 'total_gb': 14.56}, {'index': 1, 'name': 'Tesla T4', 'capability': (7, 5), 'dtype': 'torch.float16', 'free_gb': 14.46, 'total_gb': 14.56}]}
[Audio Extraction]  storage/processed/my_english_demo/audio.wav
[AudioTranscriber] Loading Whisper large-v3-turbo on cuda:0 (float16)...


config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

vocabulary.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/34.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/253 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

[MultimodalEmbedder]  SigLIP2 loaded (torch.float16)
[MultimodalEmbedder]  Released models from cuda:0
Ingestion summary: {'status': 'success', 'video_id': 'my_english_demo', 'total_clips': 153, 'total_frames': 992, 'latency_sec': 471.446, 'ocr_backend': 'easyocr', 'ocr_processed_frames': 870, 'ocr_nonempty_frames': 831, 'ocr_region_count': 12565, 'peak_vram_mb': 2946.39, 'max_observed_ram_mb': 3859.21}
Runtime after ingestion: {'cuda_available': True, 'gpu_count': 2, 'assignment': {'ingest': 'cuda:0', 'search': 'cuda:1', 'ocr': 'cuda:0'}, 'gpus': [{'index': 0, 'name': 'Tesla T4', 'capability': (7, 5), 'dtype': 'torch.float16', 'free_gb': 13.96, 'total_gb': 14.56}, {'index': 1, 'name': 'Tesla T4', 'capability': (7, 5), 'dtype': 'torch.float16', 'free_gb': 14.46, 'total_gb': 14.56}]}
Index status: {'clip_collection': 'spatio_temporal_clips', 'frame_collection': 'spatio_temporal_frames', 'video_id': 'my_english_demo', 'indexed_clips': 153, 'indexed_frames': 992, 'qdrant_connected': True,

In [28]:
# Dataset-specific OCR validation
ocr_matches = []

for item in indexer.frame_storage.values():
    payload = item.get("payload", {}) or {}
    if payload.get("video_id") != VIDEO_ID:
        continue

    if "EXPOSE 8080" in payload.get("ocr_text", "").upper():
        ocr_matches.append({
            "clip_id": payload.get("clip_id"),
            "timestamp": payload.get("timestamp"),
            "ocr_text": payload.get("ocr_text"),
            "ocr_regions": payload.get("ocr_regions", []),
        })

print("EXPOSE 8080 OCR match count:", len(ocr_matches))
print("First OCR match:", ocr_matches[0] if ocr_matches else None)

assert ocr_matches, (
    "The indexed test video does not contain an OCR match "
    "for EXPOSE 8080."
)


EXPOSE 8080 OCR match count: 66
First OCR match: {'clip_id': 'clip_6', 'timestamp': 28.94, 'ocr_text': 'Dockerfile Dockerfilel " RUN npm install COPY ENV PORT-8080 EXPOSE 8080 CMD "npm" \'start" THE DOCKERFILE', 'ocr_regions': [{'text': 'Dockerfile Dockerfilel "', 'confidence': 0.3477706633580367, 'bbox': [341, 57, 477, 77], 'bbox_norm': [533, 158, 745, 214]}, {'text': 'RUN npm install', 'confidence': 0.9715050176876402, 'bbox': [345, 91, 467, 111], 'bbox_norm': [539, 253, 730, 308]}, {'text': 'COPY', 'confidence': 0.9634279012680054, 'bbox': [343, 129, 383, 149], 'bbox_norm': [536, 358, 598, 414]}, {'text': 'ENV PORT-8080', 'confidence': 0.8881377929873073, 'bbox': [343, 167, 451, 187], 'bbox_norm': [536, 464, 705, 519]}, {'text': 'EXPOSE 8080', 'confidence': 0.670379207801701, 'bbox': [343, 205, 437, 225], 'bbox_norm': [536, 569, 683, 625]}, {'text': 'CMD', 'confidence': 0.9355266606903984, 'bbox': [345, 247, 375, 263], 'bbox_norm': [539, 686, 586, 731]}, {'text': '"npm"', 'confidenc

In [29]:
# Real video search for the first EXPOSE 8080 occurrence
expose_query = (
    "Where does EXPOSE 8080 first appear in the Dockerfile?"
)

expose_strict, expose_detailed = search_video_both(
    SearchQueryRequest(
        query=expose_query,
        video_id=VIDEO_ID,
    )
)

print("Strict response:", expose_strict.model_dump())
print("Detailed response:", expose_detailed.model_dump())

expose_spatial = expose_detailed.spatial_evidence

assert expose_spatial.get("match") is True
assert expose_spatial.get("evidence_type") == "ocr_region"
assert expose_spatial.get("bbox_norm")
assert expose_spatial.get("exact_target_match") is True
assert expose_spatial.get("matched_target") == "expose 8080"
assert "EXPOSE 8080" in expose_spatial.get(
    "matched_text",
    "",
).upper()
assert expose_detailed.temporal_relation == "none"
assert expose_detailed.relation_satisfied is True


[QueryUnderstander] deterministic strict parser
[QueryPlanner] intent=temporal_search selector=first relation=none modalities=['visual', 'ocr']
[QueryPlanner] visual='Where does EXPOSE 8080 first appear in the Dockerfile?' speech='' ocr='Where does EXPOSE 8080 first appear in the Dockerfile?'
[MultimodalEmbedder] Loading SigLIP2 on cuda:0...
[MultimodalEmbedder]  SigLIP2 loaded (torch.float16)
[MultimodalEmbedder] Loading BGE-M3 on cuda:0...
[MultimodalEmbedder]  BGE-M3 loaded
[Retriever] coarse hits: {'visual': 60, 'ocr': 60}
[MultimodalEmbedder]  Released models from cuda:0
[Reranker] Loading BGE reranker on cuda:1...


tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

[Reranker]  Loaded
[Reranker]  Released from cuda:1
[VerificationLayer] Exact OCR target selected target='expose 8080', region='EXPOSE 8080'
[VerificationLayer] Exact OCR target selected target='expose 8080', region='EXPOSE 8080'
[VerificationLayer] Exact OCR target selected target='expose 8080', region='EXPOSE 8080'
[VerificationLayer] Exact OCR target selected target='expose 8080', region='EXPOSE 8080'
[VerificationLayer] Exact OCR target selected target='expose 8080', region='EXPOSE 8080'
[VerificationLayer]  Released from cuda:1
Strict response: {'start_timestamp': '00:00:28.590', 'end_timestamp': '00:00:30.040', 'confidence_score': 0.7382}
Detailed response: {'start_timestamp': '00:00:28.590', 'end_timestamp': '00:00:30.040', 'confidence_score': 0.7382, 'search_id': 'd105b855637b4f8491229f7807264878', 'candidate_id': 'my_english_demo:clip_6', 'video_id': 'my_english_demo', 'clip_id': 'clip_6', 'speech_excerpt': "In addition, we'll look at very important advanced concepts like port

In [30]:
# Real video search for the first FROM node:12 occurrence
node_query = (
    'Where does "FROM node:12" first appear in the Dockerfile?'
)

node_strict, node_detailed = search_video_both(
    SearchQueryRequest(
        query=node_query,
        video_id=VIDEO_ID,
    )
)

print("Strict response:", node_strict.model_dump())
print("Detailed response:", node_detailed.model_dump())

node_spatial = node_detailed.spatial_evidence

assert node_spatial.get("match") is True
assert node_spatial.get("evidence_type") == "ocr_region"
assert node_spatial.get("bbox_norm")
assert node_spatial.get("exact_target_match") is True
assert node_spatial.get("matched_target") == "from node 12"
assert "FROM NODE:12" in node_spatial.get(
    "matched_text",
    "",
).upper()


[QueryUnderstander] deterministic strict parser
[QueryPlanner] intent=temporal_search selector=first relation=none modalities=['visual', 'ocr']
[QueryPlanner] visual='Where does "FROM node:12" first appear in the Dockerfile?' speech='' ocr='Where does "FROM node:12" first appear in the Dockerfile?'
[MultimodalEmbedder] Loading SigLIP2 on cuda:0...
[MultimodalEmbedder]  SigLIP2 loaded (torch.float16)
[MultimodalEmbedder] Loading BGE-M3 on cuda:0...
[MultimodalEmbedder]  BGE-M3 loaded
[Retriever] coarse hits: {'visual': 60, 'ocr': 60}
[MultimodalEmbedder]  Released models from cuda:0
[Reranker] Loading BGE reranker on cuda:1...
[Reranker]  Loaded
[Reranker]  Released from cuda:1
[VerificationLayer] Exact OCR target selected target='from node 12', region='FROM node:12'
[VerificationLayer] Exact OCR target selected target='from node 12', region='FROM node:12'
[VerificationLayer] Exact OCR target selected target='from node 12', region='FROM node:12'
[VerificationLayer] Exact OCR target sele

In [31]:
# Real search benchmark and performance report export
import json
import time
from pathlib import Path

benchmark_queries = [
    "Where does EXPOSE 8080 first appear in the Dockerfile?",
    'Where does "FROM node:12" first appear in the Dockerfile?',
    (
        'Show the screen text "PORT FORWARDING" '
        'while the speaker says "port forwarding"'
    ),
    'Show the screen text "WORKDIR /app"',
    "Show the Dockerfile in the code editor",
]

benchmark_results = []

for benchmark_query in benchmark_queries:
    started = time.perf_counter()

    try:
        strict_response, detailed_response = search_video_both(
            SearchQueryRequest(
                query=benchmark_query,
                video_id=VIDEO_ID,
            )
        )
        elapsed = time.perf_counter() - started

        benchmark_results.append({
            "query": benchmark_query,
            "status": "success",
            "wall_latency_sec": round(elapsed, 4),
            "start_timestamp": strict_response.start_timestamp,
            "end_timestamp": strict_response.end_timestamp,
            "confidence_score": strict_response.confidence_score,
            "stage_latencies": detailed_response.stage_latencies,
        })
    except Exception as exc:
        elapsed = time.perf_counter() - started

        benchmark_results.append({
            "query": benchmark_query,
            "status": "failed",
            "wall_latency_sec": round(elapsed, 4),
            "error": str(exc),
        })

successful_latencies = sorted(
    item["wall_latency_sec"]
    for item in benchmark_results
    if item["status"] == "success"
)

def percentile(values, percentile_value):
    if not values:
        return None

    position = round(
        (percentile_value / 100)
        * (len(values) - 1)
    )
    return values[position]

benchmark_summary = {
    "query_count": len(benchmark_results),
    "successful_query_count": len(successful_latencies),
    "latency_p50_sec": percentile(
        successful_latencies,
        50,
    ),
    "latency_p95_sec": (
        percentile(successful_latencies, 95)
        if len(successful_latencies) >= 5
        else None
    ),
    "latency_max_sec": (
        max(successful_latencies)
        if successful_latencies
        else None
    ),
    "results": benchmark_results,
}

output_directory = (
    Path("/kaggle/working")
    if Path("/kaggle/working").exists()
    else Path("/content")
)
output_directory.mkdir(parents=True, exist_ok=True)

benchmark_path = (
    output_directory
    / "Spatio_Temporal_Search_Benchmark.json"
)
benchmark_path.write_text(
    json.dumps(
        benchmark_summary,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

performance_report = performance_registry.arabic_markdown()
performance_report_path = (
    output_directory
    / "Spatio_Temporal_Performance_Report_AR.md"
)
performance_report_path.write_text(
    performance_report,
    encoding="utf-8",
)

print("Benchmark summary:", benchmark_summary)
print("Benchmark file:", benchmark_path)
print("Performance report file:", performance_report_path)


[QueryUnderstander] deterministic strict parser
[QueryPlanner] intent=temporal_search selector=first relation=none modalities=['visual', 'ocr']
[QueryPlanner] visual='Where does EXPOSE 8080 first appear in the Dockerfile?' speech='' ocr='Where does EXPOSE 8080 first appear in the Dockerfile?'
[MultimodalEmbedder] Loading SigLIP2 on cuda:0...
[MultimodalEmbedder]  SigLIP2 loaded (torch.float16)
[MultimodalEmbedder] Loading BGE-M3 on cuda:0...
[MultimodalEmbedder]  BGE-M3 loaded
[Retriever] coarse hits: {'visual': 60, 'ocr': 60}
[MultimodalEmbedder]  Released models from cuda:0
[Reranker] Loading BGE reranker on cuda:1...
[Reranker]  Loaded
[Reranker]  Released from cuda:1
[VerificationLayer] Exact OCR target selected target='expose 8080', region='EXPOSE 8080'
[VerificationLayer] Exact OCR target selected target='expose 8080', region='EXPOSE 8080'
[VerificationLayer] Exact OCR target selected target='expose 8080', region='EXPOSE 8080'
[VerificationLayer] Exact OCR target selected target=

## API validation and resource evaluation

The following cells validate the strict FastAPI response contract and measure the actual GPU allocation before, during, and after a real search.


In [32]:
# In-process FastAPI endpoint validation
from fastapi.testclient import TestClient

client = TestClient(app)

payload = {
    "query": 'Where does "FROM node:12" first appear in the Dockerfile?',
    "video_id": VIDEO_ID,
}

response = client.post("/search", json=payload)

print("HTTP:", response.status_code)
print("Response:", response.json())

assert response.status_code == 200

result = response.json()

assert set(result) == {
    "start_timestamp",
    "end_timestamp",
    "confidence_score",
}

assert result["start_timestamp"] < result["end_timestamp"]
assert 0.0 <= result["confidence_score"] <= 1.0

print("FastAPI search endpoint contract validated.")

[QueryUnderstander] deterministic strict parser
[QueryPlanner] intent=temporal_search selector=first relation=none modalities=['visual', 'ocr']
[QueryPlanner] visual='Where does "FROM node:12" first appear in the Dockerfile?' speech='' ocr='Where does "FROM node:12" first appear in the Dockerfile?'
[MultimodalEmbedder] Loading SigLIP2 on cuda:0...
[MultimodalEmbedder]  SigLIP2 loaded (torch.float16)
[MultimodalEmbedder] Loading BGE-M3 on cuda:0...
[MultimodalEmbedder]  BGE-M3 loaded
[Retriever] coarse hits: {'visual': 60, 'ocr': 60}
[MultimodalEmbedder]  Released models from cuda:0
[Reranker] Loading BGE reranker on cuda:1...
[Reranker]  Loaded
[Reranker]  Released from cuda:1
[VerificationLayer] Exact OCR target selected target='from node 12', region='FROM node:12'
[VerificationLayer] Exact OCR target selected target='from node 12', region='FROM node:12'
[VerificationLayer] Exact OCR target selected target='from node 12', region='FROM node:12'
[VerificationLayer] Exact OCR target sele

In [33]:
# Real GPU memory lifecycle validation
import gc
import torch

from api.main import model_manager, search_video_both
from api.schemas import SearchQueryRequest

def gpu_memory_snapshot():
    snapshot = {}

    if not torch.cuda.is_available():
        return {"device": "CPU"}

    for gpu_index in range(torch.cuda.device_count()):
        torch.cuda.synchronize(gpu_index)

        snapshot[f"gpu_{gpu_index}"] = {
            "allocated_mb": round(
                torch.cuda.memory_allocated(gpu_index) / 1024**2,
                2,
            ),
            "reserved_mb": round(
                torch.cuda.memory_reserved(gpu_index) / 1024**2,
                2,
            ),
            "max_allocated_mb": round(
                torch.cuda.max_memory_allocated(gpu_index) / 1024**2,
                2,
            ),
        }

    return snapshot


gc.collect()

for gpu_index in range(torch.cuda.device_count()):
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(gpu_index)

memory_before = gpu_memory_snapshot()
print("Memory before search:", memory_before)

strict_result, detailed_result = search_video_both(
    SearchQueryRequest(
        query=(
            'Where does "FROM node:12" first appear '
            "in the Dockerfile?"
        ),
        video_id=VIDEO_ID,
    )
)

print("Search result:", strict_result.model_dump())

memory_after_search_with_peak_counters = gpu_memory_snapshot()
print("Memory after search with peak counters:", memory_after_search_with_peak_counters)

gc.collect()

for gpu_index in range(torch.cuda.device_count()):
    torch.cuda.empty_cache()
    torch.cuda.synchronize(gpu_index)

memory_after = gpu_memory_snapshot()
print("Memory after release:", memory_after)

# Verify that model references were released.
released_models = {
    "transcriber": model_manager.transcriber is None,
    "embedder": model_manager.embedder is None,
    "query_planner": model_manager.query_planner is None,
    "reranker": model_manager.reranker is None,
    "verifier": model_manager.verifier is None,
}

print("Released model references:", released_models)
assert all(released_models.values())

# Validate actual allocated GPU memory after cleanup.
for gpu_name, metrics in memory_after.items():
    if gpu_name.startswith("gpu_"):
        assert metrics["allocated_mb"] < 500, (
            f"Unexpected allocated memory on {gpu_name}: "
            f"{metrics['allocated_mb']} MB"
        )

print("GPU memory lifecycle validated.")

Memory before search: {'gpu_0': {'allocated_mb': 18.25, 'reserved_mb': 64.0, 'max_allocated_mb': 18.25}, 'gpu_1': {'allocated_mb': 18.25, 'reserved_mb': 84.0, 'max_allocated_mb': 18.25}}
[QueryUnderstander] deterministic strict parser
[QueryPlanner] intent=temporal_search selector=first relation=none modalities=['visual', 'ocr']
[QueryPlanner] visual='Where does "FROM node:12" first appear in the Dockerfile?' speech='' ocr='Where does "FROM node:12" first appear in the Dockerfile?'
[MultimodalEmbedder] Loading SigLIP2 on cuda:0...
[MultimodalEmbedder]  SigLIP2 loaded (torch.float16)
[MultimodalEmbedder] Loading BGE-M3 on cuda:0...
[MultimodalEmbedder]  BGE-M3 loaded
[Retriever] coarse hits: {'visual': 60, 'ocr': 60}
[MultimodalEmbedder]  Released models from cuda:0
[Reranker] Loading BGE reranker on cuda:1...
[Reranker]  Loaded
[Reranker]  Released from cuda:1
[VerificationLayer] Exact OCR target selected target='from node 12', region='FROM node:12'
[VerificationLayer] Exact OCR target

In [34]:
# Positive real-video temporal condition test
positive_condition_query = (
    'Show the screen text "PORT FORWARDING" '
    'while the speaker says "port forwarding"'
)

query_planner = model_manager.get_query_planner()
positive_plan = query_planner.plan(positive_condition_query)
model_manager.release("query_planner")

print("OCR clause:", positive_plan.query_for("ocr"))
print("Speech clause:", positive_plan.query_for("speech"))

assert positive_plan.temporal_relation == "while"
assert positive_plan.relation_left_modality == "ocr"
assert positive_plan.relation_right_modality == "speech"
assert "PORT FORWARDING" in positive_plan.query_for("ocr").upper()
assert "PORT FORWARDING" in positive_plan.query_for("speech").upper()

positive_strict, positive_detailed = search_video_both(
    SearchQueryRequest(
        query=positive_condition_query,
        video_id=VIDEO_ID,
    )
)

positive_details = positive_detailed.model_dump()
speech_span = positive_details["modality_spans"].get("speech")
ocr_span = positive_details["modality_spans"].get("ocr")

assert positive_details["temporal_relation"] == "while"
assert positive_details["relation_satisfied"] is True
assert speech_span is not None
assert ocr_span is not None

overlap_start = max(speech_span[0], ocr_span[0])
overlap_end = min(speech_span[1], ocr_span[1])

assert overlap_start < overlap_end

print("Strict response:", positive_strict.model_dump())
print("Speech span:", speech_span)
print("OCR span:", ocr_span)
print("Temporal overlap:", [overlap_start, overlap_end])


[QueryUnderstander] deterministic strict parser
[QueryPlanner] intent=temporal_search selector=any relation=while modalities=['visual', 'speech', 'ocr']
[QueryPlanner] visual='Show the screen text "PORT FORWARDING"' speech='the speaker says "port forwarding"' ocr='Show the screen text "PORT FORWARDING"'
OCR clause: Show the screen text "PORT FORWARDING"
Speech clause: the speaker says "port forwarding"
[QueryUnderstander] deterministic strict parser
[QueryPlanner] intent=temporal_search selector=any relation=while modalities=['visual', 'speech', 'ocr']
[QueryPlanner] visual='Show the screen text "PORT FORWARDING"' speech='the speaker says "port forwarding"' ocr='Show the screen text "PORT FORWARDING"'
[MultimodalEmbedder] Loading SigLIP2 on cuda:0...
[MultimodalEmbedder]  SigLIP2 loaded (torch.float16)
[MultimodalEmbedder] Loading BGE-M3 on cuda:0...
[MultimodalEmbedder]  BGE-M3 loaded
[Retriever] coarse hits: {'visual': 60, 'speech': 60, 'ocr': 60}
[MultimodalEmbedder]  Released model

In [35]:
# Negative real-video temporal condition test
from fastapi import HTTPException

negative_condition_query = (
    'Show the moment when "EXPOSE 8080" is displayed '
    'while the speaker explains the exit function'
)

try:
    search_video_both(
        SearchQueryRequest(
            query=negative_condition_query,
            video_id=VIDEO_ID,
        )
    )
    raise AssertionError(
        "The system returned a result for an unsatisfied while relation."
    )
except HTTPException as exc:
    assert exc.status_code == 404
    print("Unsatisfied while relation was rejected.")
    print("API detail:", exc.detail)


[QueryUnderstander] deterministic strict parser
[QueryPlanner] intent=temporal_search selector=any relation=while modalities=['visual', 'speech', 'ocr']
[QueryPlanner] visual='Show the moment when "EXPOSE 8080" is displayed' speech='the speaker explains the exit function' ocr='Show the moment when "EXPOSE 8080" is displayed'
[MultimodalEmbedder] Loading SigLIP2 on cuda:0...
[MultimodalEmbedder]  SigLIP2 loaded (torch.float16)
[MultimodalEmbedder] Loading BGE-M3 on cuda:0...
[MultimodalEmbedder]  BGE-M3 loaded
[Retriever] coarse hits: {'visual': 60, 'speech': 60, 'ocr': 60}
[MultimodalEmbedder]  Released models from cuda:0
[Reranker] Loading BGE reranker on cuda:1...
[Reranker]  Loaded
[Reranker]  Released from cuda:1
[VerificationLayer] Exact OCR target selected target='expose 8080', region='EXPOSE 8080'
[VerificationLayer] Exact OCR target selected target='expose 8080', region='EXPOSE 8080'
[VerificationLayer] Exact OCR target selected target='expose 8080', region='EXPOSE 8080'
[Verif

In [36]:
# Manual temporal ground-truth evaluation
from evaluation.metrics import temporal_iou

def timestamp_to_seconds(timestamp):
    hours, minutes, seconds = timestamp.split(":")
    return (
        int(hours) * 3600
        + int(minutes) * 60
        + float(seconds)
    )

manual_ground_truth = {
    "expose_8080": None,
    "from_node_12": None,
    "positive_while": None,
}

predicted_intervals = {
    "expose_8080": (
        timestamp_to_seconds(expose_strict.start_timestamp),
        timestamp_to_seconds(expose_strict.end_timestamp),
    ),
    "from_node_12": (
        timestamp_to_seconds(node_strict.start_timestamp),
        timestamp_to_seconds(node_strict.end_timestamp),
    ),
    "positive_while": (
        timestamp_to_seconds(positive_strict.start_timestamp),
        timestamp_to_seconds(positive_strict.end_timestamp),
    ),
}

temporal_evaluation = {}

for test_name, prediction in predicted_intervals.items():
    truth = manual_ground_truth[test_name]

    if truth is None:
        temporal_evaluation[test_name] = {
            "predicted": prediction,
            "manual_ground_truth": None,
            "temporal_iou": None,
            "status": "Manual annotation required",
        }
        continue

    temporal_evaluation[test_name] = {
        "predicted": prediction,
        "manual_ground_truth": truth,
        "temporal_iou": round(
            temporal_iou(prediction, truth),
            4,
        ),
        "status": "Evaluated",
    }

print(temporal_evaluation)
print(
    "Enter independently annotated intervals in manual_ground_truth "
    "before reporting Temporal IoU."
)


{'expose_8080': {'predicted': (28.59, 30.04), 'manual_ground_truth': None, 'temporal_iou': None, 'status': 'Manual annotation required'}, 'from_node_12': {'predicted': (27.54, 27.94), 'manual_ground_truth': None, 'temporal_iou': None, 'status': 'Manual annotation required'}, 'positive_while': {'predicted': (30.59, 31.732), 'manual_ground_truth': None, 'temporal_iou': None, 'status': 'Manual annotation required'}}
Enter independently annotated intervals in manual_ground_truth before reporting Temporal IoU.


## Final documentation status

The notebook contains real ingestion, real vector indexing, real OCR grounding, real search queries, a positive temporal relation test, a negative temporal relation test, API contract validation, and GPU memory measurement.

Temporal IoU remains intentionally unreported until the manual_ground_truth values are entered from independent video review.

The RFC and exported performance report should be submitted as separate files with this notebook.
